# 🌸 Анализ розничного цветочного магазина на данных Бизнес.Ру
### Один универсальный блокнот: продажи · маржа · остатки · списания · поиск потерь и «подозрительных» операций · прогноз

---

**Для кого этот блокнот.** Для человека, который никогда не писал код. Всё, что нужно сделать — **нажимать ▶ (Shift+Enter) ячейку за ячейкой сверху вниз**. Менять нужно только одну ячейку: «⚙️ Настройки» (названия ваших файлов).

**Что вы получите на выходе:**
1. Понятные цифры: выручка, маржа, средний чек, лучшие и худшие товары, пики по праздникам.
2. Таблицу «**кто из сотрудников выбивается из нормы**» по 8 признакам (списания, скидки, возвраты, продажи ниже себестоимости, …).
3. Сверку «**по учёту — по факту**» (недостачи после инвентаризации).
4. ABC-XYZ анализ ассортимента, прогноз спроса, «праздничные множители» (сколько закупать к 14 февраля и 8 марта).
5. Excel-файл с результатами, который можно показать руководителю.

> ⚠️ **Важно.** Этот блокнот *не доказывает* воровство. Он показывает **отклонения от нормы** — это повод проверить (камеры, пересчёт, разговор), а не повод увольнять. Любой «красный флаг» может иметь невинное объяснение — об этом ниже есть отдельный раздел.

---

## 📑 Содержание
| № | Раздел | Зачем |
|---|--------|-------|
| 1 | Где взять данные в Бизнес.Ру | Какие отчёты скачать и как |
| 2 | Как воруют в цветочном магазине | Схемы и **в каких данных они видны** |
| 3 | Настройки и загрузка | Подключаем ваши файлы |
| 4 | Проверка качества данных | Чтобы не делать выводы по мусору |
| 5 | Общая картина бизнеса | KPI, динамика, часы пик |
| 6 | Ассортимент: ABC-XYZ | Что приносит деньги, что занимает место |
| 7 | Цены и наценка | Где мы продаём слишком дёшево |
| 8 | Остатки и «срок жизни» цветов | Заморозка денег и неизбежные списания |
| 9 | Списания | Норма и аномалии |
| 10 | Сверка с инвентаризацией | Недостачи |
| 11 | Карточка сотрудника и риск-скоринг | Кто выбивается из нормы |
| 12 | Машинное обучение: поиск аномальных смен | Алгоритм ищет странное сам |
| 13 | Подозрительные операции списком | Что конкретно проверять |
| 14 | Прогноз и праздничные множители | Сколько закупать |
| 15 | Клиенты (RFM) и корзина (что берут вместе) | Рост продаж |
| 16 | Итоги, Excel и план действий | Что делать завтра |

# 1️⃣ Где взять данные в Бизнес.Ру

> ℹ️ Я не могу зайти в ваш личный кабинет (`w746008.business.ru`), поэтому подсказываю по общей логике системы. **Названия пунктов меню у разных тарифов и версий могут слегка отличаться** — ищите по смыслу. Общее правило везде одно: открыть список или отчёт → сверху выставить **период** и **склад/магазин** → справа сверху кнопка **«Экспорт» / «Выгрузить в Excel»**. Раздел по вашей ссылке `.../supply` — это обычно учёт **поступлений (закупок у поставщиков)**.

## Какие отчёты нужны (только розница)

| Приоритет | Отчёт в Бизнес.Ру | Что внутри | Зачем нам |
|:--:|---|---|---|
| ⭐⭐⭐ **обязательно** | **Реализации / Розничные продажи (по позициям)** | Дата, номер документа, **сотрудник**, товар, кол-во, цена, скидка, сумма, себестоимость, клиент | Основа всего: выручка, маржа, скидки, возвраты, «кто продал» |
| ⭐⭐⭐ **обязательно** | **Движение товаров** (у вас `.xls`) | По каждому товару: остаток на начало, приход, расход, остаток на конец | Остатки, оборачиваемость, проверка «сходится ли баланс» |
| ⭐⭐⭐ **обязательно** | **Списания** | Дата, **сотрудник**, товар, кол-во, сумма, **причина** | Самое «опасное» место для хищений |
| ⭐⭐ очень желательно | **Инвентаризация** (акты с расхождениями: учёт vs факт) | По товару: по учёту, по факту, разница | **Единственный надёжный способ увидеть реальную недостачу** |
| ⭐⭐ очень желательно | **Поступления (Закупки)** (ваш раздел `/supply`) | Дата, поставщик, товар, кол-во, закупочная цена | Закупочная цена → точная маржа; приёмка |
| ⭐ по желанию | **Возвраты покупателей** | Если возвраты не лежат в «Реализациях» | Схема «пробил–забрал деньги–вернул» |
| ⭐ по желанию | **Клиенты / бонусные карты** | Кто покупает, как часто | RFM-анализ, повторные покупки |

## Что делать с каждым файлом — пошагово
1. Зайдите в нужный раздел (меню слева).
2. Поставьте **период**: лучше **12 месяцев** (чтобы увидеть 14 февраля, 8 марта, 1 сентября). Минимум — 3 месяца.
3. Выберите **склад / магазин** (если несколько — выгружайте отдельно или все вместе, в блокноте можно фильтровать).
4. Нажмите **«Экспорт в Excel»**. Скачается `.xls` или `.xlsx`. **Ничего в файле не правьте** — не удаляйте строки, не переименовывайте колонки.
5. Положите файлы **в ту же папку, где лежит этот блокнот**.
6. В ячейке «⚙️ Настройки» впишите имена файлов.

## Что учесть заранее (частые грабли)
* **Колонка «Сотрудник»/«Автор»/«Ответственный» должна быть в выгрузке** — иначе анализ по людям невозможен. Если в экспорте её нет — в настройках отчёта (значок колонок/«Настроить вид») включите её.
* **Себестоимость** в продажах нужна для маржи. Если её нет — блокнот возьмёт среднюю закупочную цену из «Поступлений».
* Файл `.xls` из Бизнес.Ру иногда на самом деле **HTML-страница с расширением xls**. Блокнот это умеет и откроет сам.
* В «Движении товаров» шапка таблицы **двухэтажная** (над «Количество/Сумма» стоит «Начальный остаток», «Приход» и т.д.) — блокнот склеивает её автоматически.
* Строки «Итого» блокнот удаляет сам (чтобы не посчитать дважды).
* Для **автоматической** выгрузки без ручных кликов у Бизнес.Ру есть **API** (`api.business.ru`, нужен токен, настраивается в разделе «Настройки → API»). Это следующий уровень — когда ручной режим вам надоест, сделаем отдельный скрипт.

# 2️⃣ Как воруют в цветочном магазине (и где это видно в данных)

Цветы — идеальный товар для «усушки»: товар **скоропортящийся** (всегда есть объяснение «увял»), продаётся **штучно** (9 или 11 роз — клиент не пересчитает), букет собирается «на глаз», а цена — понятие гибкое. Ниже — схемы, которые встречаются в рознице. Это нужно знать, чтобы понимать, **что искать**.

| № | Схема | Как выглядит в жизни | **Где виден след в данных** | Блок блокнота |
|:-:|---|---|---|:-:|
| 1 | **Продажа «мимо кассы»** | Клиент заплатил наличными/переводом на карту продавца, чек не пробит, товар ушёл | В продажах — **ничего** (в этом и смысл). Видно только как **недостача при инвентаризации** и как **более низкая выручка за смену** при тех же условиях | 10, 11 |
| 2 | **Фиктивное списание** («увяли», «брак») | Цветы списывают, а на деле забирают себе/перепродают/дарят знакомым | **Аномально много списаний** у одного человека, особенно дорогих позиций (пионы, розы), **круглые количества** (10, 15, 20), пустая причина, списание **в конце смены** | 9, 11, 13 |
| 3 | **Скидки «своим»** | Друзьям/родне/себе — скидка 30–50% без карты клиента | Высокая **доля скидок**, скидки **без привязки к клиенту** | 11, 13 |
| 4 | **Продажа ниже себестоимости** | Цену вручную правят в чеке: «по-свойски» | Строки, где **выручка < себестоимость** | 7, 11, 13 |
| 5 | **Пробил–забрал деньги–оформил возврат** | Чек пробит, деньги в кассу не пошли, потом «возврат» | Много **возвратов** у одного сотрудника, **возвраты без клиента**, возвраты в конце смены | 11, 13 |
| 6 | **Недовложение в букет** | В чеке «Букет 25 роз», фактически 21; разницу забирают | Не видна в продажах. Косвенно: **расход роз ниже ожидаемого** по техкартам, и **излишки** на инвентаризации по розам | 10 (см. тех. рекомендации) |
| 7 | **Пересорт на кассе** | Пробили «зелень» вместо «розы» — дешевле | **Средняя цена** по товару в чеках сотрудника ниже, чем у коллег | 7, 11 |
| 8 | **Злоупотребление бонусами/картами** | Накопление баллов на свою карту, списание бонусами «по знакомству» | Клиент с аномально частыми покупками у одного сотрудника | 15 |
| 9 | **Недовоз/недовес при приёмке** | Принимают меньше, чем в накладной, разницу забирают (в сговоре с водителем) | Расхождение **«заказано/по накладной ≠ принято»**; повышенные списания «повреждено при приёмке» | 9 |
| 10 | **Работа «на себя»** | Личные заказы через магазин: цветы магазина, упаковка, ленты — свои клиенты | Расход **упаковки и ленты** выше, чем нужно по числу проданных букетов | 9, 10 |
| 11 | **Манипуляции с инвентаризацией** | «Подгон» остатков, оприходование «из воздуха» перед проверкой | Большие корректировки сразу **перед ревизией** (оприходования + списания «в ноль») | 10 |
| 12 | **Занижение веса/количества** | Продают «10 тюльпанов» с запасом, клиент не считает | Тоже косвенно — излишки на ревизии по штучным позициям | 10 |

### ⚖️ Почему «красный флаг» ≠ вор (невинные объяснения)
* Сотрудник **работает по вечерам/в праздники** — больше свежего товара «умирает» при нём.
* Сотрудник **обслуживает постоянных клиентов** — законные скидки по карте.
* Сотрудник **принимает поставки** — естественно, при приёмке больше «повреждено».
* Сотрудник **часто стоит на кассе** — больше возвратов (клиенты возвращают).
* Новичок **плохо считает остатки** — ошибки, а не кража.

Поэтому в блокноте у каждого флага есть **«а что ещё это может значить»**, а итог — это **список для проверки**, а не приговор.

### 🛡️ Что работает лучше всего против потерь (коротко, подробно — в разделе 16)
1. **Сменная ревизия** топ-10 дорогих позиций (пересчёт при открытии/закрытии смены двумя людьми).
2. **Запрет** ручного изменения цены и ограничение скидки по правам в Бизнес.Ру.
3. **Списание — только с фото и подтверждением** второго лица (старшего флориста/владельца).
4. Скидка — **только по карте клиента**.
5. Видеонаблюдение на кассе и у холодильника.
6. **Еженедельный** просмотр этого блокнота — сам факт контроля снижает потери.

# 3️⃣ Настройки и загрузка данных

## 3.1 Подключаем библиотеки
Библиотеки — это «наборы инструментов» для Python. Просто запустите ячейку. Если выдаст ошибку `ModuleNotFoundError` — в самой первой строке раскомментируйте (уберите `#`) строку `!pip install ...`.

In [1]:
# !pip install pandas numpy matplotlib seaborn scikit-learn openpyxl xlrd lxml

import os, re, warnings, itertools
warnings.filterwarnings('ignore')
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
from IPython.display import display, Markdown

sns.set_theme(style='whitegrid')
plt.rcParams.update({'figure.figsize': (11, 5), 'font.size': 11, 'axes.titleweight': 'bold'})
pd.options.display.float_format = '{:,.2f}'.format
pd.options.display.max_columns = 60
pd.options.display.max_colwidth = 60

def say(text):
    """Печатает красивый текст (Markdown) под ячейкой."""
    display(Markdown(text))

print('✅ Библиотеки подключены')

✅ Библиотеки подключены


## 3.2 ⚙️ Настройки — **единственная ячейка, которую нужно править**

* `DEMO_MODE = True` — блокнот работает на **учебных данных** (выдуманный магазин, в котором мы заранее «посадили» двух нарушителей — проверим, найдёт ли их алгоритм). Так можно потренироваться.
* Когда будете готовы анализировать **свои** данные — поставьте `DEMO_MODE = False` и впишите **имена своих файлов** в `FILES`. Если какого-то файла нет — оставьте `None` (блокнот просто пропустит соответствующий раздел).
* Если блокнот неправильно «угадал» колонку — поправьте в `OVERRIDES` (пример внутри).

In [2]:
# ═══════════════ ⚙️ НАСТРОЙКИ ═══════════════

DEMO_MODE = True        # True = учебные данные,  False = ваши файлы из Бизнес.Ру

FILES = {                                   # None, если файла нет
    'sales':     'Реализации.xls',          # ОБЯЗАТЕЛЬНО: продажи по позициям (с сотрудником)
    'movement':  'Движение товаров.xls',    # движение товаров (остатки)
    'writeoffs': 'Списания.xls',            # списания
    'receipts':  'Поступления.xls',         # закупки у поставщиков (раздел /supply)
    'inventory': None,                      # инвентаризация (учёт vs факт), напр. 'Инвентаризация.xls'
}

# Если блокнот перепутал колонку — укажите правильную. Формат: {'что_это': 'точное название колонки в вашем файле'}
# Допустимые «что_это»: date, doc, employee, product, category, qty, price, sum, cost_sum, discount, client, op_type, reason
OVERRIDES = {
    'sales': {},          # пример: {'employee': 'Кассир', 'sum': 'Сумма со скидкой'}
    'writeoffs': {},
    'receipts': {},
}

WORK_START, WORK_END = 9, 21     # часы работы магазина (операции вне этих часов будут отмечены)
MIN_MARKUP = 1.8                 # минимальная желаемая наценка (цена/себестоимость)
FLAG_RATIO = 1.5                 # «флаг», если у сотрудника показатель в 1.5 раза хуже медианы по команде
SHELF_LIFE = {'срез': 7, 'зелен': 10, 'горш': 45, 'букет': 2}   # срок жизни по слову в названии категории (дни)
DEFAULT_SHELF_LIFE = 365         # для остальных (упаковка, игрушки) срок не ограничен
OUT_XLSX = 'результаты_анализа.xlsx'

print('✅ Настройки приняты. Режим:', 'ДЕМО-данные' if DEMO_MODE else 'ВАШИ файлы')

✅ Настройки приняты. Режим: ДЕМО-данные


## 3.3 Умная загрузка файлов (ничего менять не нужно)
Эта ячейка учит блокнот:
* открывать `.xls`, `.xlsx`, `.csv` и «поддельные xls» (HTML внутри);
* **находить строку с шапкой таблицы**, даже если над ней есть заголовок отчёта («Движение товаров за период…»);
* склеивать **двухэтажную шапку** в одну («Приход | Количество»);
* выбрасывать строки «Итого»;
* **угадывать смысл колонок** по названиям (дата, сотрудник, товар, сумма, …) — поэтому код универсальный.

In [3]:
HEADER_WORDS = ['наименован', 'товар', 'дата', 'количеств', 'кол-во', 'сумм', 'цена', 'сотрудник', 'остаток',
                'приход', 'расход', 'номер', 'артикул', 'причина', 'клиент', 'себестоим', 'поставщик', 'факт', 'учет', 'учёт']

def read_any(path):
    """Читает файл любого популярного формата в «сырую» таблицу без заголовков."""
    ext = os.path.splitext(path)[1].lower()
    if ext in ('.xls', '.xlsx', '.xlsm'):
        try:
            return pd.read_excel(path, header=None)
        except Exception as e:
            print('   Excel не открылся стандартно (' + str(e)[:60] + ') → пробую как HTML-таблицу')
            tabs = pd.read_html(path, decimal=',', thousands=' ')
            t = max(tabs, key=len)
            t.columns = [' | '.join(map(str, c)) if isinstance(c, tuple) else str(c) for c in t.columns]
            return pd.DataFrame(np.vstack([np.array(t.columns.tolist(), dtype=object), t.values.astype(object)]))
    if ext in ('.csv', '.txt'):
        for enc in ('utf-8-sig', 'cp1251', 'utf-8'):
            for sep in (';', ',', '\t'):
                try:
                    df = pd.read_csv(path, header=None, sep=sep, encoding=enc, dtype=str)
                    if df.shape[1] > 2:
                        return df
                except Exception:
                    pass
    raise ValueError('Не удалось прочитать файл: ' + path)

def promote_header(raw, max_scan=40):
    """Находит шапку таблицы (в т.ч. двухэтажную) и возвращает нормальную таблицу."""
    scores = []
    for i in range(min(max_scan, len(raw))):
        cells = raw.iloc[i].dropna().astype(str).str.lower()
        scores.append(sum(any(w in c for w in HEADER_WORDS) for c in cells))
    best = max(scores) if scores else 0
    # берём ПЕРВУЮ строку, похожую на шапку (верхний этаж), а не самую «насыщенную» (нижний этаж)
    best_i = next((i for i, sc in enumerate(scores) if sc >= max(2, 0.6 * best)), 0)
    top = raw.iloc[best_i]
    names = top.copy().astype(object)
    start = best_i + 1
    if best_i + 1 < len(raw):
        nxt = raw.iloc[best_i + 1]
        txt = nxt.dropna().astype(str)
        looks_like_header = len(txt) >= 2 and txt.str.contains(r'[A-Za-zА-Яа-я]').mean() > 0.8 \
                            and not txt.str.fullmatch(r'[\d\s.,\-]+').any()
        if looks_like_header:
            top_ff = top.ffill()
            for j in raw.columns:
                if pd.notna(nxt[j]):
                    names[j] = f'{top_ff[j]} | {nxt[j]}' if pd.notna(top_ff[j]) else str(nxt[j])
            start = best_i + 2
    names = [str(n).strip() if pd.notna(n) else f'col_{k}' for k, n in enumerate(names)]
    seen, uniq = {}, []
    for n in names:                                   # уникальные имена колонок
        seen[n] = seen.get(n, 0) + 1
        uniq.append(n if seen[n] == 1 else f'{n}_{seen[n]}')
    df = raw.iloc[start:].copy()
    df.columns = uniq
    df = df.dropna(how='all').dropna(axis=1, how='all').reset_index(drop=True)
    first_text = df.fillna('').astype(str).apply(lambda r: ' '.join(map(str, r.values)).lower(), axis=1)
    df = df[~first_text.str.contains(r'(?:^|\s)итог', regex=True)]          # убираем «Итого»
    return df.reset_index(drop=True)

def to_num(s):
    """'1 234,50' → 1234.5 ; мусор → NaN"""
    if s.dtype.kind in 'if':
        return s.astype(float)
    t = (s.astype(str).str.replace('\xa0', '', regex=False).str.replace(' ', '', regex=False)
         .str.replace(',', '.', regex=False).str.replace('%', '', regex=False))
    return pd.to_numeric(t.replace({'nan': np.nan, 'None': np.nan, '': np.nan}), errors='coerce')

# --- словарь «как может называться колонка» ---
SYN = {
    'cost_sum': ['себестоимость', 'закупочн', 'закупка'],
    'discount': ['скидка', 'скидк'],
    'employee': ['сотрудник', 'продавец', 'кассир', 'флорист', 'автор', 'ответствен', 'менеджер'],
    'client':   ['клиент', 'покупатель', 'контрагент', 'поставщик'],
    'store':    ['склад', 'магазин', 'точка', 'организац'],
    'op_type':  ['тип', 'операция', 'вид документа', 'статус'],
    'reason':   ['причина', 'комментар', 'примечан'],
    'article':  ['артикул', 'код'],
    'category': ['групп', 'категор'],
    'product':  ['наименован', 'товар', 'номенклатур', 'название'],
    'date':     ['дата', 'date', 'время'],
    'doc':      ['номер', 'чек', '№'],
    'price':    ['цена'],
    'qty':      ['количеств', 'кол-во', 'кол.', 'колич'],
    'sum':      ['сумма', 'выручка', 'итого', 'стоимость', 'оборот'],
}
FIELDS_ORDER = ['cost_sum', 'discount', 'employee', 'client', 'store', 'op_type', 'reason', 'article',
                'category', 'product', 'date', 'doc', 'price', 'qty', 'sum']

def auto_map(columns, wanted):
    used, res = set(), {}
    for f in FIELDS_ORDER:
        if f not in wanted:
            continue
        for c in columns:
            if c in used:
                continue
            if any(w in str(c).lower() for w in SYN[f]):
                res[f] = c; used.add(c); break
    return res

def build(raw, kind, wanted):
    m = auto_map(list(raw.columns), wanted)
    m.update(OVERRIDES.get(kind, {}))
    print(f'— «{kind}»: колонки определены так (проверьте глазами!):')
    for k, v in m.items():
        print(f'     {k:9s} ← «{v}»')
    missing = [k for k in wanted if k not in m]
    if missing:
        print('     не найдено (это нормально, если не критично):', missing)
    out = pd.DataFrame({k: raw[v] for k, v in m.items()})
    return out, m

print('✅ Функции загрузки готовы')

✅ Функции загрузки готовы


## 3.4 🎓 Учебные данные (генератор выдуманного магазина)
Нужны только для режима `DEMO_MODE = True`. **Если вы анализируете свои файлы — эту ячейку можно просто запустить и не читать.**

Что «зашито» в учебный магазин (мы знаем правду — потом сверимся):
* **Игорь** — завышенные списания дорогих цветов круглыми числами, много скидок «без клиента», много возвратов, продаёт ниже себестоимости, пробивает чеки после закрытия.
* **Ольга** — часть продаж «мимо кассы» (в данных их нет, но на инвентаризации образуется недостача).
* **Светлана** — много скидок, **но по картам постоянных клиентов** (законно! это «ложная тревога»).
* **Анна и Мария** — работают честно.

In [4]:
def make_demo(seed=7):
    rng = np.random.default_rng(seed)
    days = pd.date_range('2025-10-01', '2026-09-30')
    staff = ['Анна', 'Мария', 'Ольга', 'Игорь', 'Светлана']
    # название, группа, закупка за шт, наценка, популярность, типичные количества
    P = [('Роза Explorer 60 см','Срезка',55,2.4,10,[5,7,9,11,15,25]), ('Роза Freedom 70 см','Срезка',65,2.4,8,[5,9,11,15,25]),
         ('Роза Аваланш 50 см','Срезка',45,2.5,6,[5,7,11,21]), ('Роза кустовая','Срезка',90,2.3,5,[3,5,7]),
         ('Тюльпан','Срезка',30,2.6,9,[5,9,15,25,51]), ('Хризантема кустовая','Срезка',70,2.3,4,[1,3,5]),
         ('Гербера','Срезка',40,2.5,3,[3,5,7]), ('Гвоздика','Срезка',35,2.5,3,[5,11,15]),
         ('Альстромерия','Срезка',50,2.4,3,[3,5]), ('Эустома','Срезка',85,2.4,3,[3,5]),
         ('Пион','Срезка',180,2.2,3,[3,5,7]), ('Гортензия','Срезка',220,2.2,2,[1,3]),
         ('Лилия','Срезка',160,2.3,2,[1,3]), ('Подсолнух','Срезка',90,2.3,3,[1,3,5]),
         ('Эвкалипт','Зелень',60,2.5,5,[1,2,3]), ('Рускус','Зелень',35,2.5,3,[1,3,5]),
         ('Орхидея фаленопсис','Горшечные',900,1.9,2,[1]), ('Фиалка','Горшечные',150,2.2,1,[1]),
         ('Кактус микс','Горшечные',120,2.4,1,[1,2]), ('Упаковка крафт','Упаковка',15,3.0,10,[1]),
         ('Лента атласная','Упаковка',10,3.5,6,[1]), ('Букет "Нежность"','Букеты',1500,1.8,3,[1]),
         ('Букет "Классика" 25 роз','Букеты',2200,1.7,2,[1]), ('Открытка','Сопутствующие',20,4.0,5,[1]),
         ('Мягкая игрушка','Сопутствующие',450,2.0,1,[1])]
    names = [p[0] for p in P]
    pop = np.array([p[4] for p in P], float); pop /= pop.sum()
    clients = [f'Клиент {i:04d}' for i in range(1, 1501)]
    vip = clients[:25]                                  # постоянные клиенты (карты)

    def season(d):
        m = 1.0
        if d.weekday() >= 5: m *= 1.3
        if d.month in (6, 7, 8): m *= 0.8
        if d.month == 2 and d.day in (12, 13, 14): m *= {12: 1.5, 13: 2.5, 14: 5}[d.day]
        if d.month == 3 and 5 <= d.day <= 8: m *= {5: 1.5, 6: 2, 7: 3, 8: 6}[d.day]
        if d.month == 9 and d.day == 1: m *= 3
        if d.month == 12 and d.day == 31: m *= 2
        return m

    sales, wo, stolen = [], [], {n: 0 for n in names}
    doc_no = 0
    for d in days:
        n_rec = rng.poisson(14 * season(d))
        pair = list(rng.choice(staff, 2, replace=False))
        uplift = 1.2 if (d.month == 2 and d.day in (13, 14)) or (d.month == 3 and 6 <= d.day <= 8) else 1.0
        for _ in range(n_rec):
            emp = pair[rng.integers(0, 2)]
            hour = int(rng.choice([9,10,11,12,13,14,15,16,17,18,19,20], p=[.05,.08,.1,.11,.1,.07,.07,.08,.1,.12,.08,.04]))
            if emp == 'Игорь' and rng.random() < 0.08: hour = int(rng.choice([21, 22]))
            ts = d + pd.Timedelta(hours=hour, minutes=int(rng.integers(0, 60)))
            offbook = (emp == 'Ольга' and rng.random() < 0.12)
            has_card = rng.random() < 0.3
            client = (rng.choice(vip) if rng.random() < 0.5 else rng.choice(clients)) if has_card else None
            disc = 0.0
            if rng.random() < 0.04: disc = float(rng.choice([5, 10])); client = client or rng.choice(clients)
            if emp == 'Игорь' and rng.random() < 0.15: disc = float(rng.choice([20, 30, 40])); client = None
            if emp == 'Светлана' and rng.random() < 0.13: disc = 10.0; client = rng.choice(vip)
            doc_no += 1; doc = f'РЗ-{doc_no:06d}'
            is_ret = (rng.random() < (0.08 if emp == 'Игорь' else 0.012))
            picks = [int(rng.choice(len(P), p=pop)) for _l in range(int(rng.choice([1, 2, 3], p=[.5, .35, .15])))]
            if any(names[k].startswith('Роза') for k in picks) and rng.random() < 0.4:
                picks.append(names.index(str(rng.choice(['Эвкалипт', 'Лента атласная']))))
            for k in picks:
                name, grp, cost, mk, _, qopts = P[k]
                q = int(rng.choice(qopts))
                price = round(cost * mk * (uplift if grp == 'Срезка' else 1.0))
                if emp == 'Игорь' and rng.random() < 0.05: price = round(cost * 0.8)
                if offbook:
                    stolen[name] += q; continue
                sgn = -1 if is_ret else 1
                rev = sgn * q * price * (1 - disc / 100)
                sales.append(dict(Дата=ts, **{'Номер документа': doc}, Сотрудник=emp, Товар=name, **{'Группа товаров': grp},
                                  Количество=sgn * q, Цена=price, **{'Скидка, %': disc}, Сумма=round(rev, 2),
                                  **{'Себестоимость, сумма': sgn * q * cost}, Клиент=client,
                                  **{'Тип операции': 'Возврат' if is_ret else 'Продажа'}))
    S = pd.DataFrame(sales)

    # списания
    daily_q = S[S['Группа товаров'].isin(['Срезка', 'Зелень', 'Горшечные'])].groupby(
        [S['Дата'].dt.normalize(), 'Товар'])['Количество'].sum()
    shift_staff = {d: list(np.random.default_rng(int(d.strftime('%Y%m%d'))).choice(staff, 2, replace=False)) for d in days}
    cost_of = {p[0]: p[2] for p in P}; grp_of = {p[0]: p[1] for p in P}
    for (d, name), q in daily_q.items():
        k = rng.poisson(max(q, 0) * 0.05)
        if k > 0:
            emp = rng.choice(shift_staff[d])
            wo.append(dict(Дата=d + pd.Timedelta(hours=int(rng.integers(18, 21))), Сотрудник=emp, Товар=name,
                           **{'Группа товаров': grp_of[name]}, Количество=int(k), Сумма=k * cost_of[name],
                           Причина=str(rng.choice(['Увяли', 'Брак', 'Повреждены при приёмке', 'Истёк срок']))))
    for d in days:
        if 'Игорь' in shift_staff[d] and rng.random() < 0.30:
            name = str(rng.choice(['Роза Explorer 60 см', 'Роза Freedom 70 см', 'Пион', 'Гортензия', 'Лилия']))
            k = int(rng.choice([10, 15, 20, 25]))
            wo.append(dict(Дата=d + pd.Timedelta(hours=int(rng.choice([20, 21, 22]))), Сотрудник='Игорь', Товар=name,
                           **{'Группа товаров': 'Срезка'}, Количество=k, Сумма=k * cost_of[name],
                           Причина=str(rng.choice(['Брак', 'Брак', ''])) or None))
    W = pd.DataFrame(wo)

    # поступления (2 раза в неделю: пн и чт) — покрывают потребность до следующей поставки
    need = pd.concat([S.groupby([S['Дата'].dt.normalize(), 'Товар'])['Количество'].sum(),
                      -W.groupby([W['Дата'].dt.normalize(), 'Товар'])['Количество'].sum()]).groupby(level=[0, 1]).sum()
    need = need.unstack('Товар').reindex(days).fillna(0)
    sup = {'Срезка': 'Эквадор Роузес', 'Зелень': 'Флора-Опт', 'Горшечные': 'Голландия Трейд'}
    rec_days = [d for d in days if d.weekday() in (0, 3)]
    R = []
    for i, d in enumerate(rec_days):
        end = rec_days[i + 1] if i + 1 < len(rec_days) else days[-1] + pd.Timedelta(days=1)
        seg = need.loc[d:end - pd.Timedelta(days=1)].sum()
        for name in names:
            q = int(round(max(seg[name], 0) + stolen[name] * len(seg) / len(days) + rng.integers(-1, 2)))
            if q > 0:
                pr = round(cost_of[name] * (1 + rng.normal(0, 0.03)), 2)
                R.append({'Дата': d + pd.Timedelta(hours=8), 'Номер документа': f'ПС-{i:05d}',
                          'Поставщик': sup.get(grp_of[name], 'Опт-Сервис'), 'Товар': name, 'Количество': q, 'Цена': pr, 'Сумма': round(q * pr, 2)})
    R = pd.DataFrame(R)

    # движение товаров (учётные остатки: кража мимо кассы в учёте НЕ видна!)
    open_q = {n: {'Срезка': 40, 'Зелень': 30, 'Горшечные': 8, 'Букеты': 2, 'Упаковка': 100}.get(grp_of[n], 15) for n in names}
    rows, inv = [], []
    for n in names:
        o = open_q[n]; i_ = R.loc[R['Товар'] == n, 'Количество'].sum()
        sold = S.loc[S['Товар'] == n, 'Количество'].sum(); w = W.loc[W['Товар'] == n, 'Количество'].sum() if len(W) else 0
        c = o + i_ - sold - w; cs = cost_of[n]
        rows.append({'Товар': n, 'Начальный остаток | Кол-во': o, 'Начальный остаток | Сумма': o * cs,
                     'Приход | Кол-во': i_, 'Приход | Сумма': i_ * cs, 'Реализация | Кол-во': sold, 'Списание | Кол-во': w,
                     'Расход | Кол-во': sold + w, 'Расход | Сумма': (sold + w) * cs, 'Конечный остаток | Кол-во': c, 'Конечный остаток | Сумма': c * cs})
        fact = c - stolen[n] + int(rng.integers(-2, 3))                 # + естественные ошибки пересчёта
        inv.append({'Товар': n, 'Учётное количество': c, 'Фактическое количество': fact})
    return {'sales': S, 'writeoffs': W, 'receipts': R, 'movement': pd.DataFrame(rows), 'inventory': pd.DataFrame(inv)}

DEMO_TRUTH = ('Игорь — завышенные списания дорогих цветов, скидки без клиента, возвраты, продажи ниже себестоимости, '
              'операции после закрытия; Ольга — продажи «мимо кассы» (видно только по инвентаризации и по выручке за смену); '
              'Светлана — много скидок, но ЗАКОННЫХ (по картам постоянных клиентов).')
print('✅ Генератор готов')

✅ Генератор готов


## 3.5 Загружаем данные
Запустите ячейку. Если режим `DEMO_MODE = False` — будут прочитаны ваши файлы, а блокнот **напишет**, что нашёл в каждом.

In [5]:
RAW = {}
if DEMO_MODE:
    RAW = make_demo()
    print('🎓 Загружены учебные данные:', {k: v.shape for k, v in RAW.items()})
else:
    for key, path in FILES.items():
        if not path:
            continue
        if os.path.exists(path):
            RAW[key] = promote_header(read_any(path))
            print(f'✅ {key:10s} ← {path}: {RAW[key].shape[0]} строк, {RAW[key].shape[1]} колонок')
        else:
            print(f'⚠️  {key:10s}: файл «{path}» не найден в папке {os.getcwd()}')
if 'sales' not in RAW:
    raise SystemExit('❌ Нужен минимум файл продаж (sales). Проверьте FILES в настройках.')
print('\nПервые строки файла продаж — убедитесь, что это похоже на ваши данные:')
display(RAW['sales'].head(5))
if 'movement' in RAW:
    print('\nКолонки «Движения товаров» после склейки шапки:')
    print(list(RAW['movement'].columns))

🎓 Загружены учебные данные: {'sales': (9910, 12), 'writeoffs': (1424, 7), 'receipts': (2366, 7), 'movement': (25, 11), 'inventory': (25, 3)}

Первые строки файла продаж — убедитесь, что это похоже на ваши данные:


,Дата,Номер документа,Сотрудник,Товар,Группа товаров,Количество,Цена,"Скидка, %",Сумма,"Себестоимость, сумма",Клиент,Тип операции
0,2025-10-01 14:34:00,РЗ-000002,Ольга,Роза Freedom 70 см,Срезка,11,156,0.00,"1,716.00",715,NaN,Продажа
1,2025-10-01 09:02:00,РЗ-000003,Ольга,Эустома,Срезка,3,204,0.00,612.00,255,NaN,Продажа
2,2025-10-01 09:02:00,РЗ-000003,Ольга,Роза кустовая,Срезка,7,207,0.00,"1,449.00",630,NaN,Продажа
3,2025-10-01 09:02:00,РЗ-000003,Ольга,Эвкалипт,Зелень,3,150,0.00,450.00,180,NaN,Продажа
4,2025-10-01 21:37:00,РЗ-000004,Игорь,Упаковка крафт,Упаковка,1,45,0.00,45.00,15,Клиент 0021,Продажа



Колонки «Движения товаров» после склейки шапки:
['Товар', 'Начальный остаток | Кол-во', 'Начальный остаток | Сумма', 'Приход | Кол-во', 'Приход | Сумма', 'Реализация | Кол-во', 'Списание | Кол-во', 'Расход | Кол-во', 'Расход | Сумма', 'Конечный остаток | Кол-во', 'Конечный остаток | Сумма']


## 3.6 Приводим всё к единому виду
Здесь блокнот:
1. определяет, какая колонка что значит (и **печатает**, как решил — сверьте!);
2. превращает числа-«строки» в числа;
3. помечает **возвраты**;
4. считает **себестоимость**, если её нет в продажах (из закупочных цен).

In [6]:
# ---------- ПРОДАЖИ ----------
def norm_sales(raw):
    d, m = build(raw, 'sales', ['date','doc','employee','product','category','qty','price','sum','cost_sum','discount','client','op_type','store'])
    for c in ['qty', 'price', 'sum', 'cost_sum', 'discount']:
        if c in d: d[c] = to_num(d[c])
    d['date'] = pd.to_datetime(d['date'], dayfirst=True, errors='coerce')
    d = d.dropna(subset=['date', 'product']).copy()
    for c in ['employee', 'client', 'category', 'op_type', 'doc']:
        if c in d: d[c] = d[c].astype(str).str.strip().replace({'nan': np.nan, 'None': np.nan, '': np.nan})
    d['employee'] = d['employee'].fillna('Не указан') if 'employee' in d else 'Не указан'
    if 'category' not in d: d['category'] = 'Без категории'
    d['category'] = d['category'].fillna('Без категории')
    if 'qty' not in d: raise ValueError('В продажах не найдена колонка количества. Укажите её в OVERRIDES.')
    if 'client' not in d: d['client'] = np.nan
    d['revenue'] = d['sum'] if 'sum' in d else d['qty'] * d['price']
    if 'discount' in d and '%' in str(m['discount']):
        d['disc_pct'] = d['discount'].fillna(0)
    elif 'discount' in d:
        d['disc_pct'] = (d['discount'].fillna(0) / (d['revenue'].abs() + d['discount'].fillna(0)).replace(0, np.nan) * 100).fillna(0)
    else:
        d['disc_pct'] = 0.0
    d['cost'] = d['cost_sum'] if 'cost_sum' in d else np.nan
    ret = pd.Series(False, index=d.index)
    if 'op_type' in d: ret |= d['op_type'].fillna('').str.lower().str.contains('возврат')
    ret |= (d['qty'] < 0) | (d['revenue'] < 0)
    d['is_return'] = ret
    flip = d['is_return'] & (d['qty'] > 0)                # возврат записан положительным числом → делаем отрицательным
    for c in ['qty', 'revenue', 'cost']:
        d.loc[flip, c] = -d.loc[flip, c]
    if 'doc' not in d or d['doc'].isna().all():
        d['doc'] = d['date'].dt.strftime('%Y%m%d%H%M') + '_' + d['employee']
    d['day'] = d['date'].dt.normalize()
    d['hour'] = d['date'].dt.hour
    d['weekday'] = d['date'].dt.dayofweek
    d['month'] = d['date'].dt.to_period('M').astype(str)
    return d.reset_index(drop=True)

# ---------- СПИСАНИЯ ----------
def norm_writeoffs(raw):
    d, m = build(raw, 'writeoffs', ['date', 'employee', 'product', 'category', 'qty', 'sum', 'cost_sum', 'reason'])
    for c in ['qty', 'sum', 'cost_sum']:
        if c in d: d[c] = to_num(d[c])
    d['date'] = pd.to_datetime(d['date'], dayfirst=True, errors='coerce')
    d = d.dropna(subset=['date', 'product']).copy()
    d['qty'] = d['qty'].abs()
    d['wo_cost'] = (d['cost_sum'] if 'cost_sum' in d else d['sum'] if 'sum' in d else np.nan)
    d['wo_cost'] = d['wo_cost'].abs() if hasattr(d['wo_cost'], 'abs') else d['wo_cost']
    for c in ['employee', 'reason', 'category']:
        if c in d: d[c] = d[c].astype(str).str.strip().replace({'nan': np.nan, 'None': np.nan, '': np.nan})
    if 'employee' not in d: d['employee'] = 'Не указан'
    d['employee'] = d['employee'].fillna('Не указан')
    if 'reason' not in d: d['reason'] = np.nan
    d['day'] = d['date'].dt.normalize(); d['hour'] = d['date'].dt.hour
    return d.reset_index(drop=True)

# ---------- ПОСТУПЛЕНИЯ ----------
def norm_receipts(raw):
    d, m = build(raw, 'receipts', ['date', 'doc', 'client', 'product', 'qty', 'price', 'sum'])
    for c in ['qty', 'price', 'sum']:
        if c in d: d[c] = to_num(d[c])
    d['date'] = pd.to_datetime(d['date'], dayfirst=True, errors='coerce')
    d = d.dropna(subset=['date', 'product']).copy()
    d['amount'] = d['sum'] if 'sum' in d else d['qty'] * d['price']
    d['unit_cost'] = d['amount'] / d['qty'].replace(0, np.nan)
    return d.reset_index(drop=True)

# ---------- ДВИЖЕНИЕ ТОВАРОВ ----------
QTYW, SUMW = ['кол', 'колич'], ['сумм', 'стоим', 'руб']
def _pick(cols, words, kind='qty', exclude=()):
    c1 = [c for c in cols if any(w in str(c).lower() for w in words) and not any(e in str(c).lower() for e in exclude)]
    q = [c for c in c1 if any(w in str(c).lower() for w in QTYW)]
    s = [c for c in c1 if any(w in str(c).lower() for w in SUMW)]
    if kind == 'qty': res = q if q else [c for c in c1 if c not in s]
    else: res = s
    tot = [c for c in res if any(w in str(c).lower() for w in ('всего', 'итого'))]
    return tot if tot else res

def norm_movement(raw):
    cols = list(raw.columns)
    pc = auto_map(cols, ['product']).get('product')
    ac = auto_map(cols, ['category']).get('category')
    d = pd.DataFrame({'product': raw[pc].astype(str).str.strip()})
    if ac: d['category'] = raw[ac]
    spec = {'open': ['начал', 'на начало'], 'close': ['конеч', 'на конец'], 'in': ['приход', 'поступ', 'оприход'],
            'out': ['расход'], 'sold': ['реализ', 'продаж'], 'wo': ['списан'], 'inv': ['инвентар']}
    found = {}
    for key, words in spec.items():
        for kind in ('qty', 'sum'):
            cs = _pick(cols, words, kind, exclude=(['инвентар'] if key in ('in', 'out', 'wo') else []))
            if cs:
                d[f'{key}_{kind}'] = sum(to_num(raw[c]) for c in cs); found[f'{key}_{kind}'] = cs
    if 'out_qty' not in d and ('sold_qty' in d or 'wo_qty' in d):
        d['out_qty'] = d.get('sold_qty', 0) + d.get('wo_qty', 0)
    print('— «movement»: что распознано в «Движении товаров»:')
    for k, v in found.items(): print(f'     {k:10s} ← {v}')
    return d[d['product'].str.lower().ne('nan')].reset_index(drop=True)

# ---------- ИНВЕНТАРИЗАЦИЯ ----------
def norm_inventory(raw):
    cols = list(raw.columns)
    pc = auto_map(cols, ['product']).get('product')
    d = pd.DataFrame({'product': raw[pc].astype(str).str.strip()})
    def f(words, need_sum=False):
        cs = [c for c in cols if any(w in str(c).lower() for w in words)]
        cs = [c for c in cs if (any(w in str(c).lower() for w in SUMW)) == need_sum]
        return cs[0] if cs else None
    b, fa = f(['учет', 'учёт', 'книж']), f(['факт'])
    dq, ds = f(['расхожд', 'разниц', 'недостач', 'излишек']), f(['расхожд', 'разниц', 'недостач', 'излишек'], True)
    if b: d['book_qty'] = to_num(raw[b])
    if fa: d['fact_qty'] = to_num(raw[fa])
    d['diff_qty'] = (d['fact_qty'] - d['book_qty']) if (b and fa) else (to_num(raw[dq]) if dq else np.nan)
    d['diff_sum'] = to_num(raw[ds]) if ds else np.nan
    print('— «inventory»: учёт ←', b, '| факт ←', fa, '| разница ←', dq)
    return d

S = norm_sales(RAW['sales'])
W = norm_writeoffs(RAW['writeoffs']) if 'writeoffs' in RAW else None
R = norm_receipts(RAW['receipts']) if 'receipts' in RAW else None
M = norm_movement(RAW['movement']) if 'movement' in RAW else None
I = norm_inventory(RAW['inventory']) if 'inventory' in RAW else None

# ---------- эталонная закупочная цена за штуку (для маржи и оценки потерь в рублях) ----------
unit_cost = pd.Series(dtype=float)
if R is not None and len(R):
    unit_cost = R.groupby('product').apply(lambda g: g['amount'].sum() / g['qty'].sum())
elif S['cost'].notna().any():
    t = S[(~S.is_return) & S.cost.notna()]
    unit_cost = t.groupby('product').apply(lambda g: g['cost'].sum() / g['qty'].sum())
if S['cost'].isna().all() and len(unit_cost):
    S['cost'] = S['qty'] * S['product'].map(unit_cost)
    print('\nℹ️ В продажах не было себестоимости — посчитана по средней закупочной цене из «Поступлений».')
if W is not None and W['wo_cost'].isna().all() and len(unit_cost):
    W['wo_cost'] = W['qty'] * W['product'].map(unit_cost)
S['margin'] = S['revenue'] - S['cost']
HAS_COST = S['cost'].notna().any()
print(f'\n✅ Готово. Продажи: {len(S):,} строк, {S.day.min().date()} → {S.day.max().date()}, сотрудников: {S.employee.nunique()}, товаров: {S["product"].nunique()}')

— «sales»: колонки определены так (проверьте глазами!):
     cost_sum  ← «Себестоимость, сумма»
     discount  ← «Скидка, %»
     employee  ← «Сотрудник»
     client    ← «Клиент»
     op_type   ← «Тип операции»
     category  ← «Группа товаров»
     product   ← «Товар»
     date      ← «Дата»
     doc       ← «Номер документа»
     price     ← «Цена»
     qty       ← «Количество»
     sum       ← «Сумма»
     не найдено (это нормально, если не критично): ['store']
— «writeoffs»: колонки определены так (проверьте глазами!):
     employee  ← «Сотрудник»
     reason    ← «Причина»
     category  ← «Группа товаров»
     product   ← «Товар»
     date      ← «Дата»
     qty       ← «Количество»
     sum       ← «Сумма»
     не найдено (это нормально, если не критично): ['cost_sum']
— «receipts»: колонки определены так (проверьте глазами!):
     client    ← «Поставщик»
     product   ← «Товар»
     date      ← «Дата»
     doc       ← «Номер документа»
     price     ← «Цена»
     qty       ←

# 4️⃣ Проверка качества данных

> **Правило аналитика: «мусор на входе — мусор на выходе».** Прежде чем делать выводы, проверяем, что данные целые. Если здесь что-то красное — исправьте выгрузку, иначе выводы будут ложными.

In [7]:
def check(ok, good, bad):
    print(('✅ ' + good) if ok else ('❌ ' + bad))

span_days = (S.day.max() - S.day.min()).days + 1
check(span_days >= 28, f'Период {span_days} дн. — достаточно для выводов',
      f'Период всего {span_days} дн. — выводы будут шаткими (лучше 90+ дней, идеально 365)')
check(S.employee.nunique() > 1 and 'Не указан' not in S.employee.unique(), f'Сотрудников в продажах: {S.employee.nunique()}',
      'Колонка сотрудника не найдена/одна — анализ «по людям» невозможен. Добавьте её в выгрузку.')
check(HAS_COST, 'Себестоимость есть — маржа посчитается', 'Нет себестоимости — маржа и продажи ниже себестоимости недоступны. Нужны «Поступления».')
dup = S.duplicated(subset=['date', 'doc', 'product', 'qty', 'revenue']).sum()
check(dup / len(S) < 0.03, f'Явных дубликатов мало ({dup} из {len(S)} — это нормально: один товар дважды в одном чеке)', f'Найдено {dup} одинаковых строк ({dup / len(S):.0%}) — похоже, период выгрузили дважды. Проверьте файл.')
check(S.qty.eq(0).sum() == 0, 'Нулевых количеств нет', f'{S.qty.eq(0).sum()} строк с нулевым количеством')
check(S.revenue.isna().sum() == 0, 'Пропусков в суммах нет', f'{S.revenue.isna().sum()} строк без суммы')
cal = pd.date_range(S.day.min(), S.day.max())
gaps = cal.difference(S.day.unique())
check(len(gaps) <= span_days * 0.03, 'Дней без продаж мало (нормально)',
      f'{len(gaps)} дней без единой продажи — возможно, магазин не работал, а возможно, выгрузка неполная: {[str(x.date()) for x in gaps[:5]]}…')
if M is not None and {'open_qty', 'in_qty', 'out_qty', 'close_qty'} <= set(M.columns):
    bal = (M.open_qty + M.in_qty - M.out_qty - M.close_qty).abs()
    check((bal > 0.5).sum() == 0, 'Баланс «Движения товаров» сходится: начало + приход − расход = конец',
          f'У {(bal > 0.5).sum()} товаров баланс НЕ сходится — вероятно, не учтена часть колонок (перемещения, инвентаризация). Список: {M.loc[bal > 0.5, "product"].head(5).tolist()}')
if W is not None:
    check(W.wo_cost.notna().any(), f'Списаний: {len(W)}', 'В списаниях нет суммы — оценка в рублях недоступна')
print('\nПропуски по колонкам продаж (%):')
display((S[['employee', 'product', 'qty', 'revenue', 'cost', 'client']].isna().mean() * 100).round(1).to_frame('% пустых').T)

✅ Период 365 дн. — достаточно для выводов
✅ Сотрудников в продажах: 5
✅ Себестоимость есть — маржа посчитается
✅ Явных дубликатов мало (158 из 9910 — это нормально: один товар дважды в одном чеке)
✅ Нулевых количеств нет
✅ Пропусков в суммах нет
✅ Дней без продаж мало (нормально)
✅ Баланс «Движения товаров» сходится: начало + приход − расход = конец
✅ Списаний: 1424

Пропуски по колонкам продаж (%):


,employee,product,qty,revenue,cost,client
% пустых,0.00,0.00,0.00,0.00,0.00,64.90


# 5️⃣ Общая картина бизнеса

**Зачем:** прежде чем искать проблемы — понять, как выглядит «здоровый» бизнес. Здесь считаем ключевые показатели (**KPI** — Key Performance Indicators).

**Словарик:**
* **Выручка** — сколько денег принесли продажи (за вычетом возвратов).
* **Чек** — один документ продажи; **средний чек** = выручка / число чеков.
* **Маржа** — выручка минус себестоимость (то, что остаётся «на руках» до аренды, зарплат и т.д.).
* **Маржинальность, %** — маржа / выручка. Для цветочной розницы здоровая — **50–65%** (но с учётом списаний она «худеет»).

In [8]:
sales = S[~S.is_return]; rets = S[S.is_return]
n_docs = sales.doc.nunique()
kpi = {
    'Выручка (чистая, с вычетом возвратов), ₽': S.revenue.sum(),
    'Выручка от продаж, ₽': sales.revenue.sum(),
    'Возвраты, ₽': -rets.revenue.sum(),
    'Число чеков': n_docs,
    'Средний чек, ₽': sales.revenue.sum() / max(n_docs, 1),
    'Продано единиц (шт/стеблей)': sales.qty.sum(),
    'Среднее число чеков в день': n_docs / span_days,
}
if HAS_COST:
    kpi['Себестоимость проданного, ₽'] = S.cost.sum()
    kpi['Маржа, ₽'] = S.margin.sum()
    kpi['Маржинальность, %'] = S.margin.sum() / S.revenue.sum() * 100
list_amt = sales.revenue / (1 - sales.disc_pct.clip(upper=99) / 100)
kpi['Сумма предоставленных скидок, ₽'] = (list_amt - sales.revenue).sum()
kpi['Средняя скидка, % от цены без скидки'] = (list_amt - sales.revenue).sum() / list_amt.sum() * 100
if W is not None and W.wo_cost.notna().any():
    kpi['Списания (по себестоимости), ₽'] = W.wo_cost.sum()
    kpi['Списания, % от выручки'] = W.wo_cost.sum() / S.revenue.sum() * 100
    if HAS_COST:
        kpi['Маржа ПОСЛЕ списаний, ₽'] = S.margin.sum() - W.wo_cost.sum()
        kpi['Маржинальность ПОСЛЕ списаний, %'] = (S.margin.sum() - W.wo_cost.sum()) / S.revenue.sum() * 100
display(pd.Series(kpi, name='Значение').to_frame().style.format('{:,.1f}'))

AttributeError: The '.style' accessor requires jinja2

In [ ]:
fig, ax = plt.subplots(1, 2, figsize=(15, 5))
m = S.groupby('month').agg(revenue=('revenue', 'sum'), margin=('margin', 'sum'))
m['margin_pct'] = m.margin / m.revenue * 100
ax[0].bar(m.index, m.revenue / 1000, color='#e75480', alpha=.85)
ax[0].set_title('Выручка по месяцам, тыс. ₽'); ax[0].tick_params(axis='x', rotation=60)
if HAS_COST:
    a2 = ax[0].twinx(); a2.plot(m.index, m.margin_pct, color='#2c7a4b', marker='o'); a2.set_ylabel('маржинальность, %', color='#2c7a4b'); a2.grid(False)
pv = sales.pivot_table(index='weekday', columns='hour', values='revenue', aggfunc='sum').reindex(range(7)).fillna(0)
pv.index = ['Пн', 'Вт', 'Ср', 'Чт', 'Пт', 'Сб', 'Вс']
sns.heatmap(pv / 1000, cmap='RdPu', ax=ax[1], cbar_kws={'label': 'тыс. ₽'})
ax[1].set_title('Когда зарабатываем: день недели × час'); ax[1].set_xlabel('час'); ax[1].set_ylabel('')
plt.tight_layout(); plt.show()

daily_rev = S.groupby('day').revenue.sum().reindex(pd.date_range(S.day.min(), S.day.max()), fill_value=0)
top_days = daily_rev.sort_values(ascending=False).head(8)
fig, ax = plt.subplots(figsize=(15, 4))
ax.plot(daily_rev.index, daily_rev.values / 1000, color='#999', lw=1)
ax.scatter(top_days.index, top_days.values / 1000, color='#e75480', zorder=5)
for d_, v_ in top_days.items(): ax.annotate(d_.strftime('%d.%m'), (d_, v_ / 1000), textcoords='offset points', xytext=(0, 7), ha='center', fontsize=9)
ax.set_title('Выручка по дням, тыс. ₽ (розовым — 8 самых сильных дней)'); plt.tight_layout(); plt.show()

### 📖 Как читать
* **Столбцы по месяцам** показывают сезонность. В цветах она сильная: **февраль–март (14 февраля, 8 марта)** и **сентябрь (1 сентября)** — «золотые» периоды, летом — провал.
* **Тепловая карта**: чем темнее клетка — тем больше денег. Показывает, **в какие часы должны быть два флориста** и когда можно один.
* **Розовые точки** — самые прибыльные дни. Обычно это праздники; именно к ним нужно **заранее** закупать (см. раздел 14).
* **Если «Маржинальность ПОСЛЕ списаний» заметно ниже обычной** — деньги «утекают» в списания (раздел 9).

# 6️⃣ Ассортимент: ABC-XYZ анализ

**Идея в двух строках:** делим товары на группы по двум вопросам.
* **ABC — «сколько денег приносит»:** **A** — товары, дающие первые 80% выручки (звёзды), **B** — следующие 15%, **C** — «хвост» последних 5%.
* **XYZ — «насколько предсказуем спрос»** (по недельным продажам): **X** — стабильный, **Y** — колеблется, **Z** — хаотичный (для цветов — многие сезонные).

Получается матрица 3×3: **AX** — «держим всегда и в запасе», **CZ** — «кандидаты на вылет / под заказ».

In [ ]:
p = sales.groupby('product').agg(revenue=('revenue', 'sum'), qty=('qty', 'sum'), margin=('margin', 'sum')).sort_values('revenue', ascending=False)
p['share_%'] = p.revenue / p.revenue.sum() * 100
p['cum_%'] = p['share_%'].cumsum()
p['ABC'] = np.where(p['cum_%'] - p['share_%'] < 80, 'A', np.where(p['cum_%'] - p['share_%'] < 95, 'B', 'C'))

weekly = sales.groupby(['product', pd.Grouper(key='date', freq='W')]).qty.sum().unstack('product').fillna(0)
cv = (weekly.std() / weekly.mean().replace(0, np.nan))
p['CV'] = cv
p['XYZ'] = np.where(p.CV <= 0.5, 'X', np.where(p.CV <= 1.0, 'Y', 'Z'))
p['ABC_XYZ'] = p.ABC + p.XYZ
abc = p.copy()

fig, ax = plt.subplots(1, 2, figsize=(15, 5))
top = p.head(15)[::-1]
ax[0].barh(top.index, top.revenue / 1000, color=top.ABC.map({'A': '#e75480', 'B': '#f4a6c0', 'C': '#ddd'}))
ax[0].set_title('Топ-15 товаров по выручке, тыс. ₽'); 
mx = p.groupby(['ABC', 'XYZ']).size().unstack(fill_value=0).reindex(index=['A', 'B', 'C'], columns=['X', 'Y', 'Z'], fill_value=0)
sns.heatmap(mx, annot=True, fmt='d', cmap='RdPu', ax=ax[1], cbar=False)
ax[1].set_title('Матрица ABC-XYZ: сколько товаров в каждой клетке'); ax[1].set_xlabel('XYZ (предсказуемость)'); ax[1].set_ylabel('ABC (деньги)')
plt.tight_layout(); plt.show()

actions = {'AX': 'Золото: всегда в наличии, закупать регулярно, держать страховой запас', 'AY': 'Важно: следить за сезоном, закупать по прогнозу',
           'AZ': 'Деньги, но хаос: закупать под заказы/праздники, не копить', 'BX': 'Стабильный середняк: автозаказ', 'BY': 'Середняк: по прогнозу',
           'BZ': 'Мелкими партиями', 'CX': 'Стабильный «хвост»: держать минимум (нужен для полноты витрины)', 'CY': 'Под заказ/минимальный запас',
           'CZ': 'Кандидаты на вывод из ассортимента — они занимают холодильник и списываются'}
print('Что делать с каждой группой:'); display(pd.Series(actions, name='рекомендация').to_frame())
display(p[['revenue', 'qty', 'share_%', 'ABC', 'CV', 'XYZ', 'ABC_XYZ']].head(25))

### 📖 Как читать
* В левом графике **розовые** — группа A. Если среди A-товаров «Упаковка» и «Лента» — это нормально (дешёвые, но продаются в каждом чеке).
* **Правая матрица:** большинство товаров в клетках **C** — обычная ситуация («длинный хвост»). Сокращать ассортимент стоит осторожно: в цветах хвост создаёт «полноту» витрины.
* Колонка **CV** (коэффициент вариации) — чем выше, тем непредсказуемее спрос.
* ⚙️ *Что настроить:* пороги CV (0.5 и 1.0) для цветов можно смело сдвигать — спрос на цветы по природе более «рваный», чем на, скажем, хлеб.

# 7️⃣ Цены и наценка

**Наценка** = цена продажи / себестоимость. Если себестоимость 50 ₽ и продаём за 120 ₽ — наценка ×2,4 (то есть 140%).
Ищем: (1) товары с **низкой наценкой**; (2) **строки продаж ниже себестоимости** (это либо ошибка, либо акция, либо «услуга друзьям»); (3) сотрудников, которые продают **дешевле коллег** (пересорт/«свои цены»).

In [ ]:
if HAS_COST:
    pr = sales.groupby('product').agg(revenue=('revenue', 'sum'), cost=('cost', 'sum'), qty=('qty', 'sum'))
    pr['markup'] = pr.revenue / pr.cost
    pr['margin_%'] = (pr.revenue - pr.cost) / pr.revenue * 100
    pr['margin_rub'] = pr.revenue - pr.cost
    low = pr[pr.markup < MIN_MARKUP].sort_values('margin_rub', ascending=False)
    print(f'Товаров с наценкой ниже ×{MIN_MARKUP}: {len(low)} из {len(pr)}')
    display(low.head(10))

    below = sales[(sales.cost.notna()) & (sales.revenue < sales.cost - 0.01)].copy()
    below['loss'] = below.cost - below.revenue
    print(f'\n🔻 Строк продажи НИЖЕ себестоимости: {len(below)}, общий убыток: {below.loss.sum():,.0f} ₽')
    if len(below):
        display(below.groupby('employee').agg(строк=('loss', 'size'), убыток=('loss', 'sum')).sort_values('убыток', ascending=False))

    fig, ax = plt.subplots(1, 2, figsize=(15, 5))
    sns.scatterplot(data=pr.reset_index(), x='markup', y='margin_rub', size='qty', hue='margin_%', palette='RdYlGn', ax=ax[0], legend=False, sizes=(30, 300))
    ax[0].axvline(MIN_MARKUP, color='red', ls='--'); ax[0].set_title('Наценка vs заработок на товаре (размер = объём продаж)')
    ax[0].set_xlabel('наценка (×)'); ax[0].set_ylabel('маржа, ₽')
    pr['margin_%'].sort_values().head(12).plot.barh(ax=ax[1], color='#e75480'); ax[1].set_title('12 товаров с самой низкой маржинальностью, %')
    plt.tight_layout(); plt.show()
else:
    print('Нет себестоимости — раздел пропущен. Выгрузите «Поступления» или включите колонку себестоимости в продажах.')

### 📖 Как читать
* **Левый график:** товары **левее красной линии** продаются с недостаточной наценкой. Большие пузыри слева — самые «болезненные» (продаются много, зарабатываем мало). Либо поднимайте цену, либо принимайте осознанно (например, «упаковка» и «роза» как «входной» товар).
* **Продажи ниже себестоимости:** разовые строки — возможна акция/уценка увядающего товара (это нормально!). **Системные строки у одного сотрудника** — повод поговорить.
* Если в выгрузке *цена за стебель* и *цена за букет* перепутаны, наценка может выглядеть странно — проверьте единицы измерения.

# 8️⃣ Остатки и «срок жизни» цветов

**Почему это важно именно для цветов:** срезка живёт ~5–10 дней. Если остатка **хватит на 20 дней продаж** — значит, ~половина **неизбежно завянет и уйдёт в списание**. Деньги «заморожены» и тают.

Считаем:
* **Покрытие, дней** = остаток на конец периода / средние продажи в день (на сколько дней хватит запаса).
* **Срок жизни** — берём из настроек `SHELF_LIFE` по названию категории (срез — 7 дней и т.д.).
* **Риск списания** — стоимость той части остатка, которая **не успеет продаться** до увядания.
* **Мёртвый запас** — есть остаток, а продаж не было 30+ дней.

In [ ]:
stock = None
if M is not None and 'close_qty' in M:
    cat_map = S.groupby('product').category.agg(lambda x: x.mode().iat[0])
    sold_q = S.groupby('product').qty.sum()
    last_sale = sales.groupby('product').day.max()
    t = M[['product', 'close_qty'] + [c for c in ['close_sum', 'open_qty'] if c in M]].copy().set_index('product')
    if I is not None and I['diff_qty'].notna().any():          # учётный остаток может быть «виртуальным» (завышен недостачей)
        t['close_qty_учёт'] = t.close_qty
        t['close_qty'] = (t.close_qty + I.set_index('product').diff_qty.reindex(t.index).fillna(0)).clip(lower=0)
        print('ℹ️ Остаток скорректирован на результат инвентаризации (по факту, а не по учёту).')
    t['category'] = t.index.map(cat_map).fillna('Без категории')
    t['sold_qty'] = t.index.map(sold_q).fillna(0)
    t['per_day'] = t.sold_qty / span_days
    t['cover_days'] = np.where(t.per_day > 0, t.close_qty / t.per_day, np.inf)
    def shelf(cat):
        for k, v in SHELF_LIFE.items():
            if k in str(cat).lower(): return v
        return DEFAULT_SHELF_LIFE
    t['shelf_days'] = t.category.map(shelf)
    t['unit_cost'] = t.index.map(unit_cost) if len(unit_cost) else np.nan
    if 'close_sum' in t and t.close_sum.notna().any() and 'close_qty_учёт' not in t: t['stock_value'] = t.close_sum
    else: t['stock_value'] = t.close_qty * t.unit_cost
    t['excess_qty'] = (t.close_qty - t.per_day * t.shelf_days).clip(lower=0)
    t['risk_rub'] = t.excess_qty * (t.unit_cost.fillna(t.stock_value / t.close_qty.replace(0, np.nan)))
    t['days_since_sale'] = (S.day.max() - t.index.map(last_sale)).days if len(last_sale) else np.nan
    t['dead'] = (t.close_qty > 0) & ((t.days_since_sale > 30) | t.days_since_sale.isna())
    if 'open_qty' in t: t['turnover'] = t.sold_qty / ((t.open_qty + t.close_qty) / 2).replace(0, np.nan)
    stock = t
    print(f'💰 Заморожено в остатках на конец периода: {t.stock_value.sum():,.0f} ₽')
    print(f'⚠️  Из них «не успеет продаться до увядания»: {t.risk_rub.sum():,.0f} ₽ ({t.risk_rub.sum() / max(t.stock_value.sum(), 1) * 100:.0f}%)')
    print(f'💀 Мёртвый запас (нет продаж 30+ дней): {t[t.dead].stock_value.sum():,.0f} ₽, позиций: {int(t.dead.sum())}')
    cols = ['category', 'close_qty', 'stock_value', 'per_day', 'cover_days', 'shelf_days', 'risk_rub', 'days_since_sale']
    display(t.sort_values('risk_rub', ascending=False)[cols].head(12).replace(np.inf, np.nan))
    fig, ax = plt.subplots(1, 2, figsize=(15, 5))
    tt = t[t.cover_days.replace(np.inf, np.nan).notna()].copy()
    ax[0].scatter(tt.shelf_days, tt.cover_days.clip(upper=120), s=40, color='#e75480')
    ax[0].plot([0, 120], [0, 120], 'k--', lw=1); ax[0].set_xlim(0, 60); ax[0].set_ylim(0, 120)
    ax[0].set_xlabel('срок жизни товара, дней'); ax[0].set_ylabel('запаса хватит на, дней')
    ax[0].set_title('Всё, что ВЫШЕ пунктирной линии, — не успеет продаться')
    t.groupby('category').stock_value.sum().sort_values().plot.barh(ax=ax[1], color='#9b59b6'); ax[1].set_title('Где заморожены деньги, ₽')
    plt.tight_layout(); plt.show()
else:
    print('Нет «Движения товаров» с конечным остатком — раздел пропущен.')

### 📖 Как читать
* Левый график: **каждая точка — товар**. Если точка **выше пунктирной диагонали** — запаса больше, чем товар способен прожить: закупаем слишком много.
* **Риск списания, ₽** — деньги, которые с большой вероятностью превратятся в списание. Их можно сэкономить, **закупая меньшими партиями и чаще** (в Бизнес.Ру удобно смотреть остаток в начале каждой недели).
* **Важно:** остаток «на конец периода» — одна точка во времени. Для надёжности смотрите несколько периодов.
* **«Виртуальные остатки»:** если товар украден или потерян без документов, система продолжает «видеть» его на складе. Поэтому учётный остаток может быть завышен — и блокнот, если есть инвентаризация, берёт остаток **по факту**.

# 9️⃣ Списания: норма и аномалии

Списание — самое удобное для хищения: **товар исчезает «законно»**, по документу.

**Что считается нормой в цветочной рознице:** списания обычно **3–8% от закупки** (в зависимости от сезона и ассортимента). Выше ~10–12% — нужно искать причины (перезакуп, плохая поставка, холодильник, воровство).

Что смотрим:
1. Сколько всего и динамику по месяцам.
2. **Сотрудников:** сумма, число случаев, доля **круглых** количеств (10/15/20…), доля **без причины**, доля **вечерних** списаний.
3. **Товары:** какая доля от закупки списывается.
4. **Причины:** на что ссылаются.

In [ ]:
wo_emp = None
if W is not None and len(W):
    tot_cost = W.wo_cost.sum()
    purchase = R.amount.sum() if R is not None else np.nan
    print(f'Списано: {tot_cost:,.0f} ₽ по себестоимости, {len(W)} документов')
    if not np.isnan(purchase): print(f'Это {tot_cost / purchase * 100:.1f}% от закупок ({purchase:,.0f} ₽)  — норма ориентировочно 3–8%')
    print(f'Это {tot_cost / S.revenue.sum() * 100:.1f}% от выручки')

    Wd = W.copy()
    Wd['round_flag'] = (Wd.qty >= 10) & (Wd.qty % 5 == 0)
    Wd['noreason_flag'] = Wd.reason.isna()
    Wd['evening_flag'] = Wd.hour >= WORK_END - 2
    wo_emp = Wd.groupby('employee').agg(списано_руб=('wo_cost', 'sum'), случаев=('wo_cost', 'size'), средний_случай=('wo_cost', 'mean'),
                                         доля_круглых=('round_flag', 'mean'), доля_без_причины=('noreason_flag', 'mean'),
                                         доля_вечерних=('evening_flag', 'mean')).sort_values('списано_руб', ascending=False)
    display(wo_emp.style.format({'списано_руб': '{:,.0f}', 'средний_случай': '{:,.0f}', 'доля_круглых': '{:.0%}', 'доля_без_причины': '{:.0%}', 'доля_вечерних': '{:.0%}'})
            .background_gradient(cmap='Reds', subset=['списано_руб', 'доля_круглых', 'доля_без_причины']))

    fig, ax = plt.subplots(2, 2, figsize=(15, 9))
    mo = Wd.groupby(Wd.date.dt.to_period('M').astype(str)).wo_cost.sum()
    ax[0, 0].bar(mo.index, mo.values / 1000, color='#c0392b'); ax[0, 0].set_title('Списания по месяцам, тыс. ₽'); ax[0, 0].tick_params(axis='x', rotation=60)
    (wo_emp.списано_руб / 1000).sort_values().plot.barh(ax=ax[0, 1], color='#e74c3c'); ax[0, 1].set_title('Списания по сотрудникам, тыс. ₽')
    if R is not None:
        rate = (Wd.groupby('product').qty.sum() / R.groupby('product').qty.sum()).dropna().sort_values(ascending=False).head(10)[::-1] * 100
        ax[1, 0].barh(rate.index, rate.values, color='#8e44ad'); ax[1, 0].set_title('Какая доля закупки списана по товару, % (топ-10)')
    else:
        rate = (Wd.groupby('product').qty.sum() / sales.groupby('product').qty.sum()).dropna().sort_values(ascending=False).head(10)[::-1] * 100
        ax[1, 0].barh(rate.index, rate.values, color='#8e44ad'); ax[1, 0].set_title('Списано на 100 проданных, шт (топ-10)')
    rs = Wd.reason.fillna('(не указана)').value_counts().head(8)[::-1]
    ax[1, 1].barh(rs.index, rs.values, color='#f39c12'); ax[1, 1].set_title('Причины списаний (число случаев)')
    plt.tight_layout(); plt.show()
else:
    print('Файла списаний нет — раздел пропущен (а он важен! Выгрузите «Списания»).')

### 📖 Как читать таблицу по сотрудникам
| Колонка | Тревожный сигнал | Невинное объяснение |
|---|---|---|
| **списано_руб** | У одного в 2–3 раза больше других | Он принимает поставки или работает в смену, когда приходит товар |
| **доля_круглых** | Выше ~25–30% («ровно 10, 15, 20») | Привычка списывать «пучками»/«ведром» |
| **доля_без_причины** | Выше 30% | Поле причины необязательное — забывают |
| **доля_вечерних** | Выше 60% | Закрывающий смену обычно и списывает остатки дня |
| **средний_случай** | Заметно больше других | Списывает дорогие позиции из-за плохого хранения |

**Золотое правило:** один показатель — ничего не значит. **Если у человека выбиваются 3–4 показателя сразу** — это уже повод для проверки.

# 🔟 Сверка «по учёту — по факту» (недостачи)

### Главная мысль этого раздела
**Отчёт «Движение товаров» отражает только документы.** Если продавец взял цветок и не оформил продажу/списание — **в движении этого не будет**: система «думает», что цветок лежит в холодильнике. Поэтому «Движение товаров» **всегда сходится само с собой**, и **по нему одному воровство мимо документов не найти**.

**Единственный способ увидеть реальную недостачу — пересчитать остатки руками и сравнить с учётом (инвентаризация).** Разница = **недостача (минус)** или **излишек (плюс)**.

В Бизнес.Ру: акт инвентаризации → строки «по учёту / по факту / разница». Если инвентаризацию провели, но отчёта в файл вы не выгрузили — разница видна в «Движении товаров» как отдельные колонки «Списание/Оприходование по инвентаризации» (блокнот тоже попробует их найти).

> 💡 **Совет:** делайте **выборочную** инвентаризацию (10–15 самых дорогих позиций) **каждую неделю** и **сменную** (в начале и в конце смены, два человека). Тогда недостачу можно привязать к конкретной смене и человеку — а не «ко всему магазину за полгода».

In [ ]:
shrink = None
if I is not None and I['diff_qty'].notna().any():
    t = I.set_index('product').copy()
elif M is not None and ('inv_qty' in M or 'inv_sum' in M):
    t = M.set_index('product')[[c for c in ['inv_qty', 'inv_sum'] if c in M]].rename(columns={'inv_qty': 'diff_qty', 'inv_sum': 'diff_sum'})
    print('ℹ️ Взяты колонки «по инвентаризации» из «Движения товаров». Знак: минус — недостача (если у вас наоборот — поменяйте знак).')
else:
    t = None
    print('Данных инвентаризации нет. Это самое важное, чего не хватает для поиска РЕАЛЬНЫХ потерь — рекомендую выгрузить.')

if t is not None:
    t['unit_cost'] = t.index.map(unit_cost) if len(unit_cost) else np.nan
    if 'diff_sum' not in t or t.diff_sum.isna().all(): t['diff_sum'] = t.diff_qty * t.unit_cost
    t['sold_qty'] = t.index.map(sales.groupby('product').qty.sum()).fillna(0)
    t['shrink_%_от_продаж'] = np.where(t.sold_qty > 0, -t.diff_qty / t.sold_qty * 100, np.nan)
    short = t[t.diff_qty < 0].sort_values('diff_sum')
    over = t[t.diff_qty > 0]
    shrink = t
    cost_sold = S.cost.sum() if HAS_COST else np.nan
    print(f'📉 Недостача: {-short.diff_sum.sum():,.0f} ₽ по {len(short)} позициям;  📈 излишки: {over.diff_sum.sum():,.0f} ₽ по {len(over)} позициям')
    print(f'   Чистая недостача: {-t.diff_sum.sum():,.0f} ₽' + (f'  = {-t.diff_sum.sum() / cost_sold * 100:.1f}% от себестоимости проданного' if HAS_COST else ''))
    print('   Ориентир: естественные потери (ошибки пересчёта, усушка) обычно до 1–2%. Выше — ищем причину.')
    display(short[['diff_qty', 'diff_sum', 'sold_qty', 'shrink_%_от_продаж']].head(10))
    fig, ax = plt.subplots(figsize=(13, 5))
    tt = t.sort_values('diff_sum')
    ax.bar(tt.index, tt.diff_sum, color=np.where(tt.diff_sum < 0, '#c0392b', '#27ae60')); ax.tick_params(axis='x', rotation=75)
    ax.set_title('Разница «факт − учёт», ₽ (красное — недостача, зелёное — излишек)'); plt.tight_layout(); plt.show()

# дополнительные сверки внутри «Движения товаров»
if M is not None and 'sold_qty' in M:
    chk = M.set_index('product').sold_qty.to_frame('по движению').join(S.groupby('product').qty.sum().rename('по продажам'), how='outer').fillna(0)
    chk['разница'] = chk['по движению'] - chk['по продажам']
    bad = chk[chk['разница'].abs() > 0.5]
    print(f'\n🔎 Сверка «продажи в Движении товаров» vs «файл Реализаций»: расхождений по товарам — {len(bad)}')
    if len(bad):
        print('   Причины: разные периоды/склады в выгрузках, удалённые или изменённые «задним числом» документы (проверьте журнал изменений в Бизнес.Ру!).')
        display(bad.head(10))

### 📖 Как читать
* **Красные столбцы** — недостача по товару. Если недостача **концентрируется на дорогих и «модных» позициях** (розы, пионы) — это тревожнее, чем размазанная по всем мелочам (последнее — ошибки пересчёта).
* **«shrink_% от продаж»** — какую долю продаж составила недостача. До 1–2% — шум; **3% и выше** — повод разбираться.
* **Что делать с цифрой:** сопоставьте периоды между инвентаризациями с графиками смен. Недостача появилась, когда работал один и тот же человек? Это уже гипотеза для проверки по камерам.
* **Сверка продаж** (последний блок): расхождение между отчётами означает, что документы **правили задним числом** — это тоже красный флаг (ограничьте права на редактирование проведённых документов).

# 1️⃣1️⃣ Карточка сотрудника и риск-скоринг

**Главный раздел.** Для каждого сотрудника считаем **10 показателей** и сравниваем **с медианой по команде** (медиана — «типичный» сотрудник, не искажённый крайними значениями).

> Показатель «**1.0×**» — как у типичного коллеги. «**2.0×**» — вдвое хуже. Флаг 🚩 ставится, когда значение хуже типичного в **FLAG_RATIO** раз (по умолчанию 1.5).

| Показатель | Что измеряет | Схема, на которую намекает |
|---|---|---|
| `wo_pct` | Списания в % от его выручки | №2 фиктивные списания |
| `wo_round` | Доля «круглых» списаний (10, 15, 20…) | №2 |
| `wo_noreason` | Доля списаний без указания причины | №2 |
| `disc_share` | Доля скидок в сумме продаж | №3 скидки «своим» |
| `big_disc_noclient` | Доля чеков со скидкой ≥15% **без клиента** | №3 |
| `below_cost` | Убыток от продаж ниже себестоимости, % выручки | №4 |
| `return_share` | Возвраты в % от продаж | №5 «пробил–вернул» |
| `offhours_share` | Доля чеков вне рабочих часов | №5, №1 |
| `cheap_price` | Насколько его цены ниже, чем у других на тот же товар | №7 пересорт |
| `rev_per_day` | Выручка за смену (чем **ниже** — тем хуже) | №1 «мимо кассы» (слабый сигнал!) |

**Риск-балл** — сумма весов сработавших флагов (вес 2 — «серьёзнее», 1 — «слабее»).

In [ ]:
def build_scorecard(S, W):
    sl = S[~S.is_return].copy(); rt = S[S.is_return]
    g = sl.groupby('employee')
    emp = pd.DataFrame({'revenue': g.revenue.sum(), 'days': g.day.nunique(), 'receipts': g.doc.nunique()})
    emp['rev_per_day'] = emp.revenue / emp.days
    emp['avg_check'] = emp.revenue / emp.receipts
    sl['list_amt'] = sl.revenue / (1 - sl.disc_pct.clip(upper=99) / 100)
    emp['disc_share'] = (sl.list_amt - sl.revenue).groupby(sl.employee).sum() / sl.list_amt.groupby(sl.employee).sum()
    sl['big_nc'] = (sl.disc_pct >= 15) & sl.client.isna()
    emp['big_disc_noclient'] = sl.groupby(['employee', 'doc']).big_nc.max().groupby('employee').mean()
    if HAS_COST:
        loss = (sl.cost - sl.revenue).clip(lower=0).where(sl.cost.notna(), 0)
        emp['below_cost'] = loss.groupby(sl.employee).sum() / emp.revenue
    emp['return_share'] = (-rt.groupby('employee').revenue.sum() / emp.revenue).fillna(0)
    sl['off'] = (sl.hour < WORK_START) | (sl.hour >= WORK_END)
    emp['offhours_share'] = sl.groupby(['employee', 'doc']).off.max().groupby('employee').mean()
    sl['uprice'] = sl.list_amt / sl.qty.replace(0, np.nan)
    ref = sl.groupby(['product', 'month']).uprice.transform('median')
    sl['pdev'] = sl.uprice / ref - 1
    pdev = (sl.pdev * sl.revenue).groupby(sl.employee).sum() / sl.revenue.groupby(sl.employee).sum()
    emp['cheap_price'] = (-pdev).clip(lower=0)
    if W is not None and len(W):
        w = W.assign(rnd=(W.qty >= 10) & (W.qty % 5 == 0), nor=W.reason.isna())
        emp['wo_pct'] = (w.groupby('employee').wo_cost.sum() / emp.revenue).fillna(0)
        emp['wo_round'] = w.groupby('employee').rnd.mean().reindex(emp.index).fillna(0)
        emp['wo_noreason'] = w.groupby('employee').nor.mean().reindex(emp.index).fillna(0)
    return emp

emp = build_scorecard(S, W)
HIGHER_BAD = ['wo_pct', 'wo_round', 'wo_noreason', 'disc_share', 'big_disc_noclient', 'below_cost', 'return_share', 'offhours_share', 'cheap_price']
WEIGHTS = {'wo_pct': 2, 'wo_round': 1, 'wo_noreason': 1, 'disc_share': 1, 'big_disc_noclient': 2, 'below_cost': 2,
           'return_share': 2, 'offhours_share': 1, 'cheap_price': 1, 'rev_per_day': 1}
THRESH = {k: FLAG_RATIO for k in WEIGHTS}; THRESH['rev_per_day'] = 1.10          # «мимо кассы» даёт слабый сигнал → порог мягче
LABEL = {'wo_pct': 'Списания % выручки', 'wo_round': 'Круглые списания', 'wo_noreason': 'Списания без причины', 'disc_share': 'Доля скидок',
         'big_disc_noclient': 'Крупные скидки без клиента', 'below_cost': 'Продажи ниже себестоимости', 'return_share': 'Возвраты',
         'offhours_share': 'Операции вне часов работы', 'cheap_price': 'Цены ниже, чем у коллег', 'rev_per_day': 'Низкая выручка за смену'}
INNOCENT = {'wo_pct': 'принимает поставки / работает в «тяжёлые» смены', 'wo_round': 'списывает «пучками»', 'wo_noreason': 'не заполняет поле причины',
            'disc_share': 'обслуживает постоянных клиентов по картам', 'big_disc_noclient': 'акции/уценка увядающего товара',
            'below_cost': 'уценка товара, который вот-вот завянет', 'return_share': 'работает на кассе в «проблемные» часы / принимает претензии',
            'offhours_share': 'закрывает смену, оформляет документы позже', 'cheap_price': 'продаёт больше мелкого опта/акционного товара',
            'rev_per_day': 'работает в «тихие» смены (будни, лето)'}

ratio = pd.DataFrame(index=emp.index)
for k in WEIGHTS:
    if k not in emp: continue
    x = emp[k].astype(float)
    if k == 'rev_per_day':
        ratio[k] = x.median() / x.replace(0, np.nan)
    else:
        base = max(x.median(), 0.3 * x.mean(), 1e-9)
        ratio[k] = x / base
flags = pd.DataFrame({k: (ratio[k] >= THRESH[k]) & ((emp[k] > emp[k].median()) if k != 'rev_per_day' else True) for k in ratio.columns})
score = (flags * pd.Series({k: WEIGHTS[k] for k in flags.columns})).sum(axis=1)
emp['risk_score'] = score
order = score.sort_values(ascending=False).index

say('### 📋 Сводка по сотрудникам (абсолютные значения)')
show = emp.loc[order].copy()
fmt = {c: '{:.1%}' for c in HIGHER_BAD if c in show}; fmt.update({'revenue': '{:,.0f}', 'rev_per_day': '{:,.0f}', 'avg_check': '{:,.0f}', 'risk_score': '{:.0f}'})
display(show.style.format(fmt))
say('### 🌡️ Во сколько раз показатель хуже типичного коллеги (чем краснее — тем хуже)')
rr = ratio.loc[order].rename(columns=LABEL)
display(rr.style.format('{:.1f}×').background_gradient(cmap='Reds', vmin=1, vmax=3))

fig, ax = plt.subplots(figsize=(10, 4.5))
cols = ['#c0392b' if v >= 6 else '#f39c12' if v >= 3 else '#27ae60' for v in score.loc[order]]
ax.barh(list(order)[::-1], score.loc[order].values[::-1], color=cols[::-1]); ax.set_title('Риск-балл сотрудников (🟥 ≥6 · 🟧 3–5 · 🟩 <3)')
for i, v in enumerate(score.loc[order].values[::-1]): ax.text(v + .1, i, f'{v:.0f}', va='center')
plt.tight_layout(); plt.show()

say('### 🔎 Расшифровка: что сработало и что это может значить ПОМИМО воровства')
for e in order:
    hit = [k for k in flags.columns if flags.loc[e, k]]
    if hit:
        say(f'**{e}** — риск-балл **{score[e]:.0f}**')
        for k in hit:
            say(f'- 🚩 **{LABEL[k]}** — {ratio.loc[e, k]:.1f}× от типичного. *Невинное объяснение:* {INNOCENT[k]}.')
    else:
        say(f'**{e}** — флагов нет ✅')

### 📖 Как правильно использовать эту таблицу (очень важно)
1. **Сначала смотрим на сочетание флагов, а не на один.** Скидки *одни* (особенно по карте клиента) — это маркетинг. Скидки **без клиента** + продажи ниже себестоимости + возвраты — это уже узор.
2. **Риск-балл 6+ → проверка:** посмотреть конкретные операции (раздел 13), камеры, провести внезапный пересчёт его смены. **3–5 → наблюдение.** Меньше → в норме.
3. **Не наказывайте по таблице.** Сначала выясните, нет ли невинного объяснения (колонка выше).
4. **Слабый сигнал «Низкая выручка за смену»** сам по себе ничего не доказывает (смены разные), но в сочетании с **недостачей на инвентаризации** и **отсутствием сильных флагов** характерен для схемы «мимо кассы» — такие продажи в данных просто **отсутствуют**. Поэтому для их поиска нужны *сменные ревизии* и камеры.
5. Если в команде **меньше 4 человек**, «медиана» ненадёжна — относитесь к флагам как к подсказкам.

# 1️⃣2️⃣ Машинное обучение: алгоритм сам ищет «странные» смены

Выше мы **сами** придумали правила. Теперь отдадим данные алгоритму **Isolation Forest** («изолирующий лес»), который **без правил** ищет наблюдения, непохожие на остальные. Это метод **обучения без учителя** (мы не говорим, что считается нарушением).

**Как это работает на пальцах:** алгоритм много раз случайно «режет» данные. Нормальные смены похожи друг на друга и долго «не отделяются». Странную смену можно отделить от остальных за 2–3 разреза — её «аномальность» высокая.

**Единица анализа — «сотрудник-день».** Признаки — только **относительные** (доли), чтобы праздники не выглядели аномалией.

In [ ]:
iso_res = None
try:
    from sklearn.ensemble import IsolationForest
    sl = S[~S.is_return].copy()
    sl['list_amt'] = sl.revenue / (1 - sl.disc_pct.clip(upper=99) / 100)
    sl['disc_amt'] = sl.list_amt - sl.revenue
    sl['big_nc'] = ((sl.disc_pct >= 15) & sl.client.isna()).astype(int)
    sl['off'] = ((sl.hour < WORK_START) | (sl.hour >= WORK_END)).astype(int)
    sl['loss'] = (sl.cost - sl.revenue).clip(lower=0).fillna(0) if HAS_COST else 0
    ed = sl.groupby(['employee', 'day']).agg(revenue=('revenue', 'sum'), receipts=('doc', 'nunique'), disc_amt=('disc_amt', 'sum'),
                                              list_amt=('list_amt', 'sum'), loss=('loss', 'sum'), off=('off', 'mean'), big_nc=('big_nc', 'mean'))
    ed['ret'] = (-S[S.is_return].groupby(['employee', 'day']).revenue.sum()).reindex(ed.index).fillna(0)
    ed['wo'] = W.groupby(['employee', 'day']).wo_cost.sum().reindex(ed.index).fillna(0) if W is not None else 0
    ed = ed[ed.revenue > 0]
    X = pd.DataFrame({'disc_share': ed.disc_amt / ed.list_amt, 'ret_share': ed.ret / ed.revenue, 'wo_share': ed.wo / ed.revenue,
                      'loss_share': ed.loss / ed.revenue, 'off_share': ed.off, 'big_nc': ed.big_nc}).replace([np.inf, -np.inf], 0).fillna(0)
    X['wo_share'] = X.wo_share.clip(upper=3)
    iso = IsolationForest(n_estimators=300, contamination=0.03, random_state=0).fit(X)
    ed['anomaly'] = iso.predict(X) == -1
    ed['anomaly_score'] = -iso.score_samples(X)
    byemp = ed.groupby('employee').agg(смен=('anomaly', 'size'), аномальных=('anomaly', 'sum'))
    byemp['доля_аномальных_смен'] = byemp.аномальных / byemp.смен
    iso_res = byemp
    display(byemp.sort_values('доля_аномальных_смен', ascending=False).style.format({'доля_аномальных_смен': '{:.1%}'}))
    fig, ax = plt.subplots(figsize=(9, 4))
    (byemp.доля_аномальных_смен * 100).sort_values().plot.barh(ax=ax, color='#8e44ad'); ax.set_title('Доля «аномальных» смен у сотрудника, %')
    ax.axvline(3, color='gray', ls='--'); ax.text(3.1, 0, 'в среднем по магазину ~3%', fontsize=9); plt.tight_layout(); plt.show()
    say('**10 самых «странных» смен** (смотрите, какие показатели в них необычны):')
    top_idx = ed[ed.anomaly].sort_values('anomaly_score', ascending=False).head(10).index
    top = X.loc[top_idx].join(ed[['revenue', 'receipts', 'anomaly_score']])
    display(top[['revenue', 'receipts', 'disc_share', 'ret_share', 'wo_share', 'loss_share', 'off_share', 'anomaly_score']].style.format(
        {'revenue': '{:,.0f}', 'disc_share': '{:.0%}', 'ret_share': '{:.0%}', 'wo_share': '{:.0%}', 'loss_share': '{:.0%}', 'off_share': '{:.0%}', 'anomaly_score': '{:.2f}'}))
except ImportError:
    print('Не установлен scikit-learn: раздел пропущен (pip install scikit-learn)')

### 📖 Как читать
* **Доля аномальных смен** около 3% у всех — норма (мы так задали). **Если у человека 10–30%** — его смены систематически «не такие, как у всех».
* Алгоритм **не знает, что хорошо, а что плохо** — он нашёл «непохожее». Это может быть и воровство, и хаотичный день с большой акцией. Поэтому смотрим **таблицу «странных» смен** и ищем в ней **повторяющиеся причины**.
* Это пример идеи из Data Science: **не пытайся написать все правила — дай алгоритму найти необычное**. В компаниях так ищут мошенничество по банковским картам.

# 1️⃣3️⃣ Подозрительные операции — конкретный список для проверки

Здесь — **конкретные документы**, которые можно открыть в Бизнес.Ру по номеру и проверить: есть ли чек на кассе, что на камере, кто клиент. Таблицы отсортированы по сумме — сначала самое дорогое.

In [ ]:
R_ = {'date': 'Дата', 'doc': 'Документ', 'employee': 'Сотрудник', 'product': 'Товар', 'qty': 'Кол-во', 'revenue': 'Сумма', 'disc_pct': 'Скидка %',
      'client': 'Клиент', 'cost': 'Себест.', 'loss': 'Убыток', 'disc_amt': 'Скидка ₽', 'reason': 'Причина', 'wo_cost': 'Списано ₽'}
def show_tbl(df, title, n=12):
    say(f'#### {title}')
    if len(df) == 0: print('   — не найдено ✅'); return
    display(df.head(n).rename(columns=R_).style.format(precision=0, na_rep='—'))

sl = S[~S.is_return].copy()
sl['list_amt'] = sl.revenue / (1 - sl.disc_pct.clip(upper=99) / 100); sl['disc_amt'] = sl.list_amt - sl.revenue
c1 = sl[(sl.disc_pct >= 15) & sl.client.isna()].sort_values('disc_amt', ascending=False)
show_tbl(c1[['date', 'doc', 'employee', 'product', 'qty', 'revenue', 'disc_pct', 'disc_amt']], f'1) Скидка ≥15% без клиента (всего {len(c1)} строк, скидок на {c1.disc_amt.sum():,.0f} ₽)')
if HAS_COST:
    c2 = sl[sl.cost.notna() & (sl.revenue < sl.cost - 0.01)].assign(loss=lambda d: d.cost - d.revenue).sort_values('loss', ascending=False)
    show_tbl(c2[['date', 'doc', 'employee', 'product', 'qty', 'revenue', 'cost', 'loss']], f'2) Продажи ниже себестоимости (всего {len(c2)} строк, убыток {c2.loss.sum():,.0f} ₽)')
c3 = S[S.is_return & S.client.isna()].assign(revenue=lambda d: -d.revenue).sort_values('revenue', ascending=False)
show_tbl(c3[['date', 'doc', 'employee', 'product', 'qty', 'revenue']], f'3) Возвраты без указания клиента (всего {len(c3)}, на {c3.revenue.sum():,.0f} ₽)')
c4 = sl[(sl.hour < WORK_START) | (sl.hour >= WORK_END)].sort_values('revenue', ascending=False)
show_tbl(c4[['date', 'doc', 'employee', 'product', 'qty', 'revenue']], f'4) Продажи вне рабочих часов ({WORK_START}:00–{WORK_END}:00): {len(c4)} строк')
if W is not None and len(W):
    c5 = W[(W.qty >= 10) & (W.qty % 5 == 0) & (W.reason.isna() | (W.hour >= WORK_END - 2))].sort_values('wo_cost', ascending=False)
    show_tbl(c5[['date', 'employee', 'product', 'qty', 'wo_cost', 'reason']], f'5) Крупные «круглые» списания без причины или вечером ({len(c5)} шт., на {c5.wo_cost.sum():,.0f} ₽)')

### 📖 Как пользоваться
1. Берёте **номер документа** → открываете в Бизнес.Ру → смотрите кто оформил, что в комментарии.
2. Сверяете с **кассой** (был ли чек) и **камерами** на это время.
3. **Не обвиняйте по списку.** Сначала узнайте контекст: возможно, был корпоративный клиент или уценка.
4. Если одни и те же фамилии/клиенты повторяются — переходите к разговору или к внутренней проверке.

# 1️⃣4️⃣ Прогноз спроса и «праздничные множители»

**Зачем:** главное решение флориста-закупщика — **сколько заказать**. Мало — упущенная выручка в пик; много — списания. Прогноз помогает найти баланс.

**Метод (специально простой и честный):**
1. «Профиль дня недели»: берём **медиану** выручки за последние 8 недель по каждому дню недели (медиана не боится выбросов-праздников).
2. Умножаем на **тренд** (последние 4 недели против предыдущих 4, ограничен 0.8–1.25).
3. **Проверяем себя** (backtest): прогнозируем уже прошедшие 2-недельные периоды и сравниваем с реальностью — считаем ошибку **WAPE** (чем меньше, тем лучше; <25% для рознёй цветов хорошо).
4. Отдельно считаем **праздничные множители**: во сколько раз выручка 14 февраля/8 марта/1 сентября выше обычного дня.

In [ ]:
daily = S.groupby('day').revenue.sum().reindex(pd.date_range(S.day.min(), S.day.max()), fill_value=0)

def forecast(series, horizon=14, weeks=8):
    h = series.iloc[-weeks * 7:]
    prof = h.groupby(h.index.dayofweek).median()
    last, prev = series.iloc[-28:].median(), series.iloc[-56:-28].median() if len(series) >= 56 else series.iloc[-28:].median()
    trend = float(np.clip(last / prev, 0.8, 1.25)) if prev > 0 else 1.0
    idx = pd.date_range(series.index[-1] + pd.Timedelta(days=1), periods=horizon)
    return pd.Series([prof[d.dayofweek] * trend for d in idx], index=idx)

if len(daily) >= 98:
    d_err, d_base, w_err, w_base = [0, 0], [0, 0], [0, 0], [0, 0]       # [сумма ошибок, сумма факта]
    for k in range(6):                                             # 6 проверочных окон по 14 дней
        end = len(daily) - 14 * (k + 1)
        if end < 56: break
        hist, fact = daily.iloc[:end], daily.iloc[end:end + 14]
        f = forecast(hist, len(fact)); f.index = fact.index
        b = pd.Series(hist.iloc[-14:].mean(), index=fact.index)
        d_err[0] += abs(f - fact).sum(); d_base[0] += abs(b - fact).sum(); d_err[1] += fact.sum(); d_base[1] += fact.sum()
        fw, bw, aw = f.resample('W').sum(), b.resample('W').sum(), fact.resample('W').sum()
        w_err[0] += abs(fw - aw).sum(); w_base[0] += abs(bw - aw).sum(); w_err[1] += aw.sum()
    wape, wape_b = d_err[0] / d_err[1] * 100, d_base[0] / d_err[1] * 100
    wape_w, wape_wb = w_err[0] / w_err[1] * 100, w_base[0] / w_err[1] * 100
    print(f'Ошибка по ДНЯМ (WAPE):   наш метод {wape:.0f}%  |  «наивный» (среднее за 2 недели) {wape_b:.0f}%')
    print(f'Ошибка по НЕДЕЛЯМ (WAPE): наш метод {wape_w:.0f}%  |  «наивный» {wape_wb:.0f}%')
    print('   → ' + ('наш метод точнее наивного ✅' if wape_w < wape_wb else 'метод не лучше наивного ⚠️ — для закупок опирайтесь на прошлогодние объёмы и праздничные множители'))
    print('   (Ориентир: по дням 25–40% — типичная ошибка для розницы цветов, по неделям — 10–20%.)')
    fc = forecast(daily, 14)
    fig, ax = plt.subplots(figsize=(14, 4.5))
    ax.plot(daily.index[-70:], daily.values[-70:] / 1000, color='#555', label='факт'); ax.plot(fc.index, fc.values / 1000, color='#e75480', marker='o', label='прогноз')
    ax.set_title('Выручка в день, тыс. ₽: последние 10 недель и прогноз на 14 дней'); ax.legend(); plt.tight_layout(); plt.show()
    print(f'Ожидаемая выручка на ближайшие 14 дней (без праздников): {fc.sum():,.0f} ₽')
else:
    print('Для прогноза нужно минимум ~100 дней данных.')

# ---- праздничные множители ----
HOL = {'14 февраля': (2, 14), '8 марта': (3, 8), '7 марта': (3, 7), '1 сентября': (9, 1), '31 декабря': (12, 31)}
rows = []
for name, (mo, dd) in HOL.items():
    for y in sorted(set(daily.index.year)):
        try: d0 = pd.Timestamp(y, mo, dd)
        except ValueError: continue
        if d0 not in daily.index: continue
        win = daily.loc[d0 - pd.Timedelta(days=28): d0 + pd.Timedelta(days=28)]
        win = win[(abs((win.index - d0).days) > 4) & ((win.index.dayofweek >= 5) == (d0.dayofweek >= 5))]
        if len(win) >= 4 and win.median() > 0:
            rows.append({'праздник': name, 'год': y, 'выручка_в_день_праздника': daily[d0], 'обычный_такой_же_день': win.median(), 'множитель': daily[d0] / win.median()})
hol = pd.DataFrame(rows)
if len(hol):
    say('#### 🎉 Праздничные множители (во сколько раз праздничный день сильнее обычного)')
    display(hol.style.format({'выручка_в_день_праздника': '{:,.0f}', 'обычный_такой_же_день': '{:,.0f}', 'множитель': '{:.1f}×'}))
    base_now = daily.iloc[-28:].median()
    plan = hol.groupby('праздник').множитель.mean().to_frame('средний_множитель')
    plan['ожидаемая_выручка_дня'] = plan.средний_множитель * base_now
    say(f'#### 📆 План к ближайшим праздникам (обычный день сейчас ≈ {base_now:,.0f} ₽)')
    display(plan.style.format({'средний_множитель': '{:.1f}×', 'ожидаемая_выручка_дня': '{:,.0f}'}))
    # что именно продаётся в праздник
    best = hol.sort_values('множитель', ascending=False).iloc[0]
    d0 = pd.Timestamp(int(best.год), *HOL[best.праздник])
    q_day = sales[sales.day == d0].groupby('product').qty.sum()
    q_norm = sales.groupby('product').qty.sum() / span_days
    tab = pd.DataFrame({'продано_в_праздник': q_day, 'в_обычный_день': q_norm}).dropna()
    tab['во_сколько_раз_больше'] = tab.продано_в_праздник / tab.в_обычный_день
    say(f'#### 🌹 Что и сколько продавалось {best.праздник} {int(best.год)} (запас к следующему году — отсюда)')
    display(tab.sort_values('продано_в_праздник', ascending=False).head(8).style.format('{:,.1f}'))
else:
    print('В данных нет праздничных дат с достаточным окружением — множители не посчитаны.')

### 📖 Как читать и использовать
* **WAPE** — «в среднем на сколько процентов мы ошибаемся». Если метод проигрывает «наивному» — не используйте его для закупок, просто смотрите на множители.
* **Множитель 5×** значит: если обычно вы продаёте на 40 тыс. ₽, то 14 февраля — на 200 тыс. ₽. Закупать нужно **к пику**, учитывая, что **цветы в пик закупаются заранее и дороже** (поэтому в блокноте цена продажи в праздник выше).
* **Таблица «что продавалось»** — готовая основа заказа поставщику на следующий год: берёте прошлогодние объёмы × прогнозируемый рост.
* Дальнейшее развитие (когда накопится 2–3 года): модели **Prophet / SARIMA / градиентный бустинг** с признаками «праздник», «погода», «день недели», «акция».

# 1️⃣5️⃣ Клиенты (RFM) и «что берут вместе»

## RFM — разделяем клиентов на группы
Три вопроса о каждом клиенте:
* **R**ecency — как давно покупал? (чем недавно — тем лучше)
* **F**requency — как часто?
* **M**onetary — на какую сумму?

По ним клиент получает оценки 1–5, и мы выделяем сегменты: **«Лучшие», «Новые», «Уходят», «Потерянные», «Середнячки»**. Каждому — свой подход (лучшим — персональное внимание, «уходящим» — напоминание о датах, «потерянным» — акция «вернись»).

> RFM работает только для покупок, **привязанных к клиенту (карта/телефон)**. Если таких мало — ниже блокнот так и напишет.

In [ ]:
cl = sales[sales.client.notna()]
share_known = cl.revenue.sum() / sales.revenue.sum()
print(f'Доля выручки с привязкой к клиенту: {share_known:.0%}  (ниже ~30% — RFM будет неполным; приучайте сотрудников спрашивать карту/телефон!)')
rfm = None
if cl.client.nunique() >= 20:
    now = S.day.max() + pd.Timedelta(days=1)
    rfm = cl.groupby('client').agg(last=('day', 'max'), F=('doc', 'nunique'), M=('revenue', 'sum'))
    rfm['R_days'] = (now - rfm['last']).dt.days
    rfm['R'] = pd.qcut(rfm.R_days.rank(method='first', ascending=False), 5, labels=[1, 2, 3, 4, 5]).astype(int)
    rfm['Fs'] = pd.qcut(rfm.F.rank(method='first'), 5, labels=[1, 2, 3, 4, 5]).astype(int)
    rfm['Ms'] = pd.qcut(rfm.M.rank(method='first'), 5, labels=[1, 2, 3, 4, 5]).astype(int)
    def seg(r):
        if r.R >= 4 and r.Fs >= 4: return 'Лучшие'
        if r.R >= 4 and r.Fs <= 2: return 'Новые / недавние'
        if r.R <= 2 and r.Fs >= 4: return 'Уходят (были лучшими!)'
        if r.R <= 2 and r.Fs <= 2: return 'Потерянные'
        return 'Середнячки'
    rfm['segment'] = rfm.apply(seg, axis=1)
    sg = rfm.groupby('segment').agg(клиентов=('M', 'size'), выручка=('M', 'sum'), ср_чеков=('F', 'mean'), дней_с_покупки=('R_days', 'mean')).sort_values('выручка', ascending=False)
    sg['доля_выручки_%'] = sg.выручка / sg.выручка.sum() * 100
    display(sg.style.format({'выручка': '{:,.0f}', 'ср_чеков': '{:.1f}', 'дней_с_покупки': '{:.0f}', 'доля_выручки_%': '{:.0f}'}))
    rep = (rfm.F >= 2).mean()
    print(f'\n🔁 Доля клиентов с 2+ покупками (повторные): {rep:.0%}.  Для цветочной розницы 20–35% — хороший результат.')
    fig, ax = plt.subplots(figsize=(8, 4)); sg['доля_выручки_%'].sort_values().plot.barh(ax=ax, color='#e75480'); ax.set_title('Вклад сегментов в выручку (по известным клиентам), %'); plt.tight_layout(); plt.show()
else:
    print('Мало идентифицированных клиентов (<20) — RFM пропущен.')

# ---- корзина: что берут вместе ----
say('### 🛒 Что покупают вместе (анализ корзины)')
cart = sales.groupby('doc')['product'].apply(lambda x: sorted(set(x)))
cart = cart[cart.str.len() >= 1]
n = len(cart)
topP = sales.groupby('product').doc.nunique().sort_values(ascending=False).head(15).index
sup_ = {p: sum(p in c for c in cart) / n for p in topP}
pairs = []
for a, b in itertools.combinations(topP, 2):
    both = sum((a in c) and (b in c) for c in cart)
    if both >= 20:
        pairs.append({'товар_1': a, 'товар_2': b, 'чеков_вместе': both, 'доля_чеков_%': both / n * 100, 'lift': (both / n) / (sup_[a] * sup_[b])})
pairs = pd.DataFrame(pairs)
if len(pairs):
    display(pairs.sort_values('lift', ascending=False).head(10).style.format({'доля_чеков_%': '{:.1f}', 'lift': '{:.2f}'}))
else:
    print('Недостаточно совместных покупок.')

### 📖 Как читать
* **Сегмент «Уходят (были лучшими!)»** — самый ценный для работы: это ваши постоянные клиенты, которые перестали приходить. Позвоните/напишите (например, к дате, которую вы знаете из прошлых покупок).
* **Lift** в корзине: показывает, во сколько раз два товара покупают **вместе чаще, чем если бы они выбирались независимо**. **Lift > 1.2** — товары «тянут» друг друга: выкладывайте рядом, делайте готовые наборы («розы + эвкалипт + упаковка»), предлагайте дополнение на кассе.
* **Lift ≈ 1** — связи нет. **Lift < 1** — товары скорее взаимозаменяемы.
* Это упрощённая версия метода **Market Basket Analysis / ассоциативные правила (Apriori)**.

# 1️⃣6️⃣ Итоги, Excel и план действий

In [ ]:
lines = []
lines.append(f'**Период:** {S.day.min().date()} — {S.day.max().date()} ({span_days} дн.). **Выручка:** {S.revenue.sum():,.0f} ₽, **чеков:** {n_docs:,}, **средний чек:** {sales.revenue.sum() / n_docs:,.0f} ₽.')
if HAS_COST: lines.append(f'**Маржинальность:** {S.margin.sum() / S.revenue.sum() * 100:.0f}%' + (f' (после списаний — {(S.margin.sum() - W.wo_cost.sum()) / S.revenue.sum() * 100:.0f}%).' if W is not None else '.'))
if W is not None: lines.append(f'**Списания:** {W.wo_cost.sum():,.0f} ₽ = {W.wo_cost.sum() / S.revenue.sum() * 100:.1f}% выручки' + (f' и {W.wo_cost.sum() / R.amount.sum() * 100:.1f}% закупок (норма 3–8%).' if R is not None else '.'))
if shrink is not None: lines.append(f'**Недостача по инвентаризации:** {-shrink.diff_sum.sum():,.0f} ₽ (по документам она не привязана к человеку; привязать можно только сменной ревизией и камерами).')
if stock is not None: lines.append(f'**Деньги в остатках:** {stock.stock_value.sum():,.0f} ₽, из них под риском увядания {stock.risk_rub.sum():,.0f} ₽.')
hi = [e for e in order if emp.loc[e, 'risk_score'] >= 6]; mid = [e for e in order if 3 <= emp.loc[e, 'risk_score'] < 6]
lines.append('**🟥 Приоритет проверки (риск ≥6):** ' + (', '.join(f'{e} ({emp.loc[e, "risk_score"]:.0f})' for e in hi) if hi else 'нет'))
lines.append('**🟧 Под наблюдением (3–5):** ' + (', '.join(f'{e} ({emp.loc[e, "risk_score"]:.0f})' for e in mid) if mid else 'нет'))
say('## 🧾 Краткие выводы\n\n' + '\n\n'.join('- ' + l for l in lines))

if DEMO_MODE:
    say('---\n### 🎓 Сверка с правдой (только в учебном режиме)\n' + DEMO_TRUTH +
        '\n\n**Проверьте:** нашёл ли блокнот Игоря (должен быть сверху), и увидели ли недостачу, связанную с Ольгой (в разделе 10 — по инвентаризации). '
        'Светлана может получить 1–2 флага по скидкам — это «ложная тревога»: скидки законные (по картам), поэтому **флаги — повод проверить, а не доказательство**.')

sheets = {'Сотрудники': emp.loc[order].reset_index(), 'Риск_множители': rr.reset_index(), 'ABC_XYZ': abc.reset_index()}
if wo_emp is not None: sheets['Списания_по_людям'] = wo_emp.reset_index()
if shrink is not None: sheets['Недостачи'] = shrink.reset_index()
if stock is not None: sheets['Остатки_риск'] = stock.reset_index().replace(np.inf, np.nan)
if iso_res is not None: sheets['Аномальные_смены'] = iso_res.reset_index()
if rfm is not None: sheets['RFM_клиенты'] = rfm.reset_index()
try:
    with pd.ExcelWriter(OUT_XLSX, engine='openpyxl') as xw:
        for name, df_ in sheets.items(): df_.to_excel(xw, sheet_name=name[:31], index=False)
    print(f'\n💾 Результаты сохранены в файл «{OUT_XLSX}» (листов: {len(sheets)}) — он лежит рядом с блокнотом.')
except Exception as e:
    print('Не удалось сохранить Excel:', e)

## 🛡️ План действий (что делать с выводами)

### Быстро (на этой неделе)
1. **Включить в Бизнес.Ру ограничения по правам:** запрет ручного изменения цены в продаже; максимальная скидка по роли; запрет редактирования/удаления проведённых документов.
2. **Скидка только по карте клиента** (иначе — не пробивается). Объясните команде: это не недоверие, а правило.
3. **Поле «причина списания» — обязательное**, плюс фото увядшего товара в рабочем чате.
4. **Списание подтверждает второй человек** (старший флорист/владелец) — особенно дорогие позиции.
5. **Выгрузите инвентаризацию** (если делали) и подключите в `FILES['inventory']`.

### Регулярно (каждую неделю)
6. **Выборочная ревизия 10–15 самых дорогих позиций** (розы, пионы, гортензия, орхидеи). Пересчитывает **не тот, кто работал**.
7. **Сменная ревизия:** пересчёт при открытии и закрытии смены двумя людьми, запись в журнал. Так недостача привязывается к смене.
8. **Запускать этот блокнот раз в неделю** и смотреть динамику риск-баллов. Сам факт «аналитика работает» снижает потери.

### Организационно
9. **Разделение ролей:** кто принимает товар ≠ кто списывает ≠ кто пересчитывает.
10. **Мотивация:** часть бонуса — от низкого уровня списаний и нулевой недостачи (общая на смену), а не только от продаж.
11. **Камеры:** на кассе и у холодильника; запись хранить 30+ дней.
12. **Техкарты букетов:** фиксированный состав и количество стеблей — тогда «недовложение» видно по расходу материала.
13. **Закупка:** чаще и меньшими партиями для быстро вянущих позиций (см. раздел 8), заранее — под праздники (раздел 14).

---

## 🧠 Какие знания Data Science / аналитики мы здесь применили (и что ещё можно)

| Область | Что сделали в блокноте | Куда расти |
|---|---|---|
| **Подготовка данных (ETL)** | Универсальная загрузка xls/csv/HTML, поиск шапки, склейка двухэтажных заголовков, проверки качества | Автовыгрузка из **API Бизнес.Ру** по расписанию |
| **BI / KPI** | Выручка, чек, маржа, тепловые карты | Дашборд в **Power BI / Yandex DataLens / Metabase** поверх Excel/БД |
| **Продуктовая аналитика** | ABC-XYZ, RFM, повторные покупки, корзина (lift) | **Когортный анализ** (удержание по месяцам первой покупки), LTV |
| **Финансовая аналитика** | Наценка, маржа после списаний, продажи ниже себестоимости | Юнит-экономика на букет, точка безубыточности |
| **Управление запасами** | Покрытие, срок жизни, мёртвый запас | Модель оптимального заказа (**newsvendor**: баланс «недозакуп / перезакуп») |
| **Антифрод (forensics)** | Правила + робастные сравнения с медианой + Isolation Forest | Модель с «учителем» (когда накопятся подтверждённые случаи), сетевой анализ связей «сотрудник–клиент» |
| **Прогнозирование** | Профиль недели × тренд, backtest, праздничные множители | **Prophet / SARIMA / бустинг** с погодой и акциями |
| **Статистика** | Медиана, коэффициент вариации, z-логика, WAPE | A/B-тесты акций и цен (разные цены/витрины по точкам или неделям) |
| **Оптимизация цен** | Выявление «слишком дешёвых» товаров | Эластичность спроса: как меняется объём при изменении цены |

---

## 📚 Мини-словарь
* **KPI** — ключевой показатель эффективности.
* **Маржа** — выручка минус себестоимость. **Наценка** — цена / себестоимость.
* **Медиана** — «серединное» значение (половина данных выше, половина ниже); не зависит от выбросов.
* **CV (коэффициент вариации)** — разброс относительно среднего; мера «непредсказуемости».
* **ABC / XYZ** — классификация товаров по деньгам / по стабильности спроса.
* **RFM** — сегментация клиентов по давности, частоте и сумме покупок.
* **Lift** — во сколько раз вероятность совместной покупки выше случайной.
* **Isolation Forest** — алгоритм поиска аномалий без разметки.
* **WAPE** — взвешенная ошибка прогноза в процентах.
* **Shrinkage (усушка)** — потери товара, которые не объясняются продажами и списаниями.
* **Backtest** — проверка метода на прошлом: «как бы он сработал, если бы мы им пользовались тогда».

---
*Если какая-то колонка определилась неверно или ваш отчёт выглядит иначе — пришлите первые строки файла (без личных данных), и правило распознавания можно дополнить.*